# EPANET-AI (Этап 2): Дистилляция знаний от Сильной Модели-Учителя (`Teacher -> Student`) и обучение `Qwen3.5-4B`

Этот ноутбук — продолжение `EPANET_AI_Pipeline.ipynb`. Он подхватывает папки **`data_cleaned`** и **`artifacts_epanet_ai`** и полностью реализует **дистилляцию знаний от большой модели-Учителя в компактную локальную модель `Qwen/Qwen3.5-4B`** без консольных команд.

### Как работает конвейер дистилляции в этом ноутбуке:
1. **Сжатие таблиц в `facts_<город>.json` (Клетка 3):** сырые матрицы `R0..R1439` (десятки МБ) не отправляются в API — в Учителя уходит только компактный проверенный JSON-паспорт (3–8 КБ) без утечки гигабайтных баз.
2. **Генерация пула сложных инженерных задач для Учителя (Клетка 4):** по каждому городу создаются 9–15 разноплановых практических сценариев (аналитическая записка главному инженеру, выдача техусловий застройщику, диагностика гидроудара и застоя воды по Walski, настройка ЧРП/PRV насосов, вопросы диспетчера, ловушки против выдумывания отсутствующих данных).
3. **Разметка Сильной Моделью-Учителем с автоисправлением цифр (`Critique & Retry`, Клетка 5):**
   - большой Учитель (`Qwen3.5-397B` / `Qwen3.5-122B` / `DeepSeek-V3.2` / `Claude Opus` / `GPT-5` через OpenRouter или локальная `Qwen3.5-35B` в Ollama) пишет глубокие живые инженерные разборы;
   - каждый ответ Учителя мгновенно проверяется детектором чисел: если Учитель округлил или посчитал в уме число, которого нет в `facts_inline`, ноутбук **автоматически отправляет ему замечание со списком лишних чисел и заставляет переписать ответ строго по фактам**;
   - все проверенные ответы кэшируются на диск (`teacher_distilled_cache.jsonl`), чтобы при повторном запуске не тратить токены дважды.
4. **Сборка чистого `train.jsonl` / `val.jsonl` (Клетка 6) + Обучение табличного ML (Клетка 7) + QLoRA-дообучение локальной `Qwen/Qwen3.5-4B` (Клетки 8–11).**

## Клетка 1. Подключение папок `data_cleaned`, `artifacts_epanet_ai` и выбор модели ИИ
Положите этот ноутбук рядом с `EPANET_AI_Pipeline.ipynb` (в корень вашей папки `EPANET-AI`) — обе папки подхватятся автоматически.

In [ ]:
import os
from pathlib import Path

CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "data_cleaned").exists() or (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data_cleaned").exists() or (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Входные папки от первого ноутбука EPANET_AI_Pipeline.ipynb
DATA_CLEANED_DIR = PROJECT_ROOT / "data_cleaned"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts_epanet_ai"

# Рабочая папка дистилляции, датасетов JSONL и весов обученных моделей
TRAINING_DIR = PROJECT_ROOT / "training_epanet_ai"
FACTS_DIR = TRAINING_DIR / "facts"
DATASETS_DIR = TRAINING_DIR / "datasets"
WEB_BATCHES_DIR = TRAINING_DIR / "web_chat_batches"
MODELS_DIR = TRAINING_DIR / "models"

for _d in (FACTS_DIR, DATASETS_DIR, WEB_BATCHES_DIR, MODELS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# 1. НАСТРОЙКИ БОЛЬШОЙ МОДЕЛИ-УЧИТЕЛЯ (TEACHER) В OPENROUTER ДЛЯ РАЗМЕТКИ ДАННЫХ
# ==============================================================================
TEACHER_BASE_URL = "https://openrouter.ai/api/v1"

# Вставьте ваш ключ OpenRouter sk-or-v1-... в кавычки ниже (или положите его в файл .openrouter_key рядом с ноутбуком):
_KEY_FILE = PROJECT_ROOT / ".openrouter_key"
TEACHER_API_KEY = (
    ""  # <--- ВСТАВЬТЕ ВАШ КЛЮЧ СЮДА: "sk-or-v1-..."
    or os.environ.get("OPENROUTER_API_KEY", "")
    or os.environ.get("EPANET_TEACHER_API_KEY", "")
    or (_KEY_FILE.read_text(encoding="utf-8").strip() if _KEY_FILE.exists() else "")
)

# Как только ключ указан — онлайн-дистилляция включается автоматически!
RUN_TEACHER_DISTILLATION = bool(TEACHER_API_KEY.strip())

# Проверенные ID моделей Qwen в OpenRouter (никаких галочек на сайте OpenRouter включать не нужно!):
# 1) "qwen/qwen3.5-397b-a17b"     — РЕКОМЕНДУЕМЫЙ УЧИТЕЛЬ (~$0.45–$0.90 за все 6 городов; родной флагман для Qwen3.5-4B)
# 2) "qwen/qwen3.5-122b-a10b"     — Младший флагман Qwen 3.5 (~$0.25 за все 6 городов)
# 3) "qwen/qwen3.7-plus"          — Свежий коммерческий Qwen (~$0.20 за все 6 городов)
# 4) "qwen/qwen3-235b-a22b-2507"  — Сверхбюджетный гигант Instruct (~$0.11 за все 6 городов)
TEACHER_MODEL_ID = "qwen/qwen3.5-397b-a17b"

# Отключаем скрытые токены <think> у гибридного Учителя: ответ в 3 раза быстрее и в 2.5 раза дешевле!
TEACHER_ENABLE_REASONING = False
TEACHER_MAX_BUDGET_USD = 3.00             # Жёсткий предохранитель расходов за запуск (защита вашего баланса $10)
MAX_TEACHER_RETRIES_ON_HALLUCINATION = 2  # Автоисправление Учителя, если он выдумал цифру не из facts_inline
INCLUDE_SEED_EXAMPLES_WITH_TEACHER = True # Добавлять базовые проверочные примеры к ответам Учителя

# ==============================================================================
# 2. НАСТРОЙКИ МАЛЕНЬКОЙ ЛОКАЛЬНОЙ МОДЕЛИ-УЧЕНИКА (STUDENT) ДЛЯ QLoRA-ОБУЧЕНИЯ
# ==============================================================================
# 1) "Qwen/Qwen3.5-4B"                  — ОСНОВНОЙ ВЫБОР (~3–6 ГБ VRAM в 4-bit, словарь 248K, 262K контекст)
# 2) "Qwen/Qwen3.5-9B"                  — ФЛАГМАН (если есть 12–16+ ГБ VRAM: RTX 3060 12GB / 4070 / 4090)
# 3) "Qwen/Qwen3.5-2B"                  — СВЕРХЛЁГКАЯ (2–4 ГБ VRAM)
# 4) "ai-sage/GigaChat3-10B-A1.8B-bf16" — ОТЕЧЕСТВЕННАЯ MoE (если заказчику обязательна российская база; 12+ ГБ VRAM)
SELECTED_MODEL_ID = "Qwen/Qwen3.5-4B"
ENABLE_THINKING_MODE = False

TOP_K_CRITICAL_ITEMS = 8
VAL_SPLIT_RATIO = 0.20
RANDOM_SEED = 42
INSTALL_LLM_LIBRARIES = False
NUM_TRAIN_EPOCHS = 3          # Число эпох QLoRA (3–4 оптимально против переобучения; при необходимости можно поставить 5)
CONTINUE_FROM_SAVED_LORA = True  # Если адаптер уже есть, дообучать поверх него, а не с нуля!
RUN_LLM_TRAINING_NOW = False

print(f"Корень проекта EPANET-AI : {PROJECT_ROOT.resolve()}")
print(f"Модель-Учитель (Teacher) : {TEACHER_MODEL_ID} @ {TEACHER_BASE_URL} (онлайн-разметка: {RUN_TEACHER_DISTILLATION})")
print(f"Лимит защиты бюджета     : ${TEACHER_MAX_BUDGET_USD:.2f} за сессию")
print(f"Модель-Ученик  (Student) : {SELECTED_MODEL_ID}")


## Клетка 2. Подключение библиотек и глубокая автопроверка видеокарты (NVIDIA / AMD / Intel + CUDA)
Напрямую опрашивает драйвер ОС (`nvidia-smi` и реестр/WMI Windows), чтобы увидеть вашу реальную физическую видеокарту даже в тех случаях, когда:
- библиотека `torch` ещё не установлена;
- на Windows по умолчанию поставилась **CPU-версия PyTorch (`torch+cpu`)**, из-за которой `torch.cuda.is_available()` ошибочно выдаёт `False`;
- в ноутбуке/ПК сразу **две видеокарты** (встроенная Intel/AMD + дискретная NVIDIA RTX/GTX).

> 💡 **Если видеокарта NVIDIA найдена, но в строке PyTorch написано «установлен CPU-вариант» или «не установлен»:** просто поменяйте в первой строчке кода ниже `AUTO_FIX_GPU_PYTORCH = False` на **`True`** и нажмите `Shift + Enter` — ноутбук сам скачает правильную CUDA-сборку PyTorch под ваш драйвер без консоли!


In [ ]:
# Поставьте True и нажмите Shift+Enter, если физическая карта NVIDIA найдена, а PyTorch стоит без CUDA (+cpu) или не установлен:
AUTO_FIX_GPU_PYTORCH = False

import csv
import html
import importlib.metadata
import importlib.util
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import unicodedata
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from IPython.display import HTML, Markdown, display
except ImportError:
    HTML = Markdown = lambda x: x
    display = print

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 1000)


def check_pkg(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def _run_hidden_cmd(cmd: list[str], timeout_s: float = 8.0) -> str:
    """Запускает системную утилиту без всплывающего чёрного окна CMD в Windows."""
    kwargs: dict[str, Any] = {
        "capture_output": True,
        "text": True,
        "encoding": "utf-8",
        "errors": "replace",
        "timeout": timeout_s,
    }
    if platform.system() == "Windows":
        kwargs["creationflags"] = getattr(subprocess, "CREATE_NO_WINDOW", 0x08000000)
    res = subprocess.run(cmd, **kwargs)
    return res.stdout.strip() if res.returncode == 0 else ""


def detect_physical_gpus() -> list[dict[str, Any]]:
    """Находит все физические видеокарты ПК напрямую через nvidia-smi и Windows WMI/Реестр (не зависит от PyTorch!)."""
    found: list[dict[str, Any]] = []
    seen_names: set[str] = set()

    # 1. Прямая проверка NVIDIA через nvidia-smi (в PATH и стандартных путях драйвера Windows)
    smi_candidates = [shutil.which("nvidia-smi")]
    if platform.system() == "Windows":
        smi_candidates.extend([
            r"C:\Windows\System32\nvidia-smi.exe",
            r"C:\Program Files\NVIDIA Corporation\NVSMI\nvidia-smi.exe",
        ])
    smi_bin = next((p for p in smi_candidates if p and Path(p).exists()), None)

    max_driver_cuda: str | None = None
    if smi_bin:
        header_out = _run_hidden_cmd([smi_bin])
        m_cuda = re.search(r"CUDA Version:\s*(\d+\.\d+)", header_out)
        if m_cuda:
            max_driver_cuda = m_cuda.group(1)

        csv_out = _run_hidden_cmd([
            smi_bin,
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader,nounits",
        ])
        for line in csv_out.splitlines():
            parts = [p.strip() for p in line.split(",")]
            if len(parts) >= 2 and parts[0]:
                gname = parts[0]
                try:
                    vram_gb = round(float(parts[1]) / 1024.0, 2)
                except ValueError:
                    vram_gb = 0.0
                drv = parts[2] if len(parts) >= 3 else ""
                found.append({
                    "name": gname,
                    "vendor": "NVIDIA",
                    "vram_gb": vram_gb,
                    "driver": drv,
                    "driver_cuda": max_driver_cuda,
                    "is_discrete": True,
                })
                seen_names.add(gname.lower())

    # 2. Опрос всех видеоадаптеров Windows через PowerShell CIM + 64-битный объём VRAM из реестра
    # (видит NVIDIA даже без nvidia-smi в PATH, а также дискретные карты AMD Radeon RX и Intel Arc)
    if platform.system() == "Windows":
        ps_script = (
            "[Console]::OutputEncoding = [System.Text.Encoding]::UTF8; "
            "$regMem = @{}; "
            "Get-ItemProperty 'HKLM:\\SYSTEM\\CurrentControlSet\\Control\\Class\\{4d36e968-e325-11ce-bfc1-08002be10318}\\0*' "
            "-ErrorAction SilentlyContinue | ForEach-Object { "
            "  $d = $_.'DriverDesc'; $q = $_.'HardwareInformation.qwMemorySize'; "
            "  if ($d -and $q) { $regMem[$d] = [int64]$q } "
            "}; "
            "Get-CimInstance Win32_VideoController -ErrorAction SilentlyContinue | ForEach-Object { "
            "  $n = $_.Name; "
            "  $r = if ($n -and $regMem.ContainsKey($n)) { $regMem[$n] } else { [int64]$_.AdapterRAM }; "
            "  [PSCustomObject]@{ name=$n; vram_bytes=$r; driver=$_.DriverVersion } "
            "} | ConvertTo-Json -Compress"
        )
        ps_bin = shutil.which("powershell") or "powershell"
        raw_json = _run_hidden_cmd([ps_bin, "-NoProfile", "-NonInteractive", "-Command", ps_script], timeout_s=10.0)
        if raw_json:
            try:
                parsed = json.loads(raw_json)
                items = parsed if isinstance(parsed, list) else [parsed]
                for it in items:
                    gname = str(it.get("name") or "").strip()
                    if not gname or gname.lower() in seen_names:
                        continue
                    low = gname.lower()
                    if any(skip in low for skip in ("microsoft basic", "remote desktop", "parsec", "virtual", "spacedesk", "citrix")):
                        continue
                    vbytes = int(it.get("vram_bytes") or 0)
                    vram_gb = round(vbytes / (1024**3), 2) if vbytes > 0 else 0.0
                    if "nvidia" in low or "geforce" in low or "quadro" in low or "rtx" in low or "gtx" in low:
                        vendor, is_disc = "NVIDIA", True
                    elif "amd" in low or "radeon" in low:
                        is_disc = ("rx " in low) or ("pro " in low) or ("xt" in low) or (vram_gb >= 4.0)
                        vendor = "AMD"
                    elif "intel" in low:
                        is_disc = ("arc" in low) or (vram_gb >= 6.0)
                        vendor = "Intel"
                    else:
                        vendor, is_disc = "Other", vram_gb >= 4.0
                    found.append({
                        "name": gname,
                        "vendor": vendor,
                        "vram_gb": vram_gb,
                        "driver": str(it.get("driver") or ""),
                        "driver_cuda": max_driver_cuda if vendor == "NVIDIA" else None,
                        "is_discrete": is_disc,
                    })
                    seen_names.add(low)
            except Exception:
                pass

    # 3. Резервная проверка lspci на Linux (если nvidia-smi не сработал)
    elif platform.system() == "Linux" and not found and shutil.which("lspci"):
        lspci_out = _run_hidden_cmd(["lspci"])
        for line in lspci_out.splitlines():
            if any(k in line for k in ("VGA compatible controller:", "3D controller:", "Display controller:")):
                gname = line.split(":", 2)[-1].strip()
                low = gname.lower()
                if any(skip in low for skip in ("vmware", "virtio", "cirrus", "qxl", "bochs", "matrox", "aspeed")):
                    continue
                vendor = "NVIDIA" if "nvidia" in low else ("AMD" if ("amd" in low or "radeon" in low) else "Intel")
                found.append({
                    "name": gname,
                    "vendor": vendor,
                    "vram_gb": 0.0,
                    "driver": "",
                    "driver_cuda": None,
                    "is_discrete": vendor == "NVIDIA" or "arc" in low or "rx " in low,
                })

    # Сортируем так, чтобы дискретная карта (особенно NVIDIA) ВСЕГДА стояла первой (актуально для ноутбуков с 2 GPU!)
    def _gpu_rank(g: dict[str, Any]) -> tuple[int, int, float]:
        v_score = 3 if g["vendor"] == "NVIDIA" else (2 if g["is_discrete"] else 1)
        return (v_score, int(bool(g["is_discrete"])), float(g["vram_gb"]))

    found.sort(key=_gpu_rank, reverse=True)
    return found


def choose_pytorch_cuda_wheel_tag(driver_cuda_str: str | None) -> str:
    """Подбирает актуальную сборку PyTorch CUDA (cu126 / cu128) с колёсами под все версии Python 3.10–3.14."""
    if not driver_cuda_str:
        return "cu126"
    try:
        ver = float(driver_cuda_str)
    except ValueError:
        return "cu126"
    if ver >= 12.8:
        return "cu128"
    # Для всех карт RTX 20xx/30xx/40xx на современных драйверах оптимальна сборка cu126 (поддерживает Python 3.10–3.14)
    if ver >= 12.0:
        return "cu126"
    return "cu118"


PHYSICAL_GPUS = detect_physical_gpus()
PRIMARY_NVIDIA_GPU = next((g for g in PHYSICAL_GPUS if g["vendor"] == "NVIDIA"), None)
HAS_NVIDIA_HW = PRIMARY_NVIDIA_GPU is not None
DETECTED_CUDA_WHEEL_TAG = choose_pytorch_cuda_wheel_tag(
    PRIMARY_NVIDIA_GPU.get("driver_cuda") if PRIMARY_NVIDIA_GPU else None
)

# Проверяем версию установленного torch по метаданным ДО импорта (чтобы Windows не заблокировал c10.dll при автопереустановке!)
try:
    TORCH_META_VERSION = importlib.metadata.version("torch")
except importlib.metadata.PackageNotFoundError:
    TORCH_META_VERSION = None

if AUTO_FIX_GPU_PYTORCH and HAS_NVIDIA_HW:
    needs_cuda_reinstall = (TORCH_META_VERSION is None) or ("+cpu" in TORCH_META_VERSION.lower())
    if not needs_cuda_reinstall and "torch" not in sys.modules:
        needs_cuda_reinstall = True
    if needs_cuda_reinstall:
        if "torch" in sys.modules:
            print(
                "⚠️ Модуль torch уже был загружен в оперативную память этого ядра Python (Windows держит его .dll файлы). "
                "Сейчас попробую установить CUDA-версию; если появится ошибка доступа [WinError 5], нажмите кнопку "
                "'Restart' (Перезапустить ядро) вверху ноутбука и сразу выполните Клетку 2!"
            )
        idx_url = f"https://download.pytorch.org/whl/{DETECTED_CUDA_WHEEL_TAG}"
        print(
            f"⏳ Устанавливаю PyTorch с поддержкой вашей видеокарты {PRIMARY_NVIDIA_GPU['name']} "
            f"(сборка {DETECTED_CUDA_WHEEL_TAG} из {idx_url})... Подождите 1–3 минуты."
        )
        try:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install",
                "--upgrade", "--force-reinstall", "torch",
                "--index-url", idx_url,
            ])
            print(
                "✅ CUDA-версия PyTorch успешно установлена! Если ниже статус ещё не обновился — "
                "нажмите 'Restart' (Перезапуск ядра) в панели ноутбука, верните AUTO_FIX_GPU_PYTORCH = False и запустите Клетку 2."
            )
        except Exception as install_err:
            print(
                f"⚠️ Не удалось перезаписать DLL на лету ({install_err}). "
                "Решение за 5 секунд: нажмите 'Restart' (Перезапустить ядро) вверху ноутбука и запустите ТОЛЬКО Клетку 2 с AUTO_FIX_GPU_PYTORCH = True!"
            )

HAS_SKLEARN = check_pkg("sklearn")
HAS_TORCH = check_pkg("torch")
HAS_TRANSFORMERS = check_pkg("transformers")
HAS_PEFT = check_pkg("peft")
HAS_TRL = check_pkg("trl")
HAS_BITSANDBYTES = check_pkg("bitsandbytes")
HAS_UNSLOTH = check_pkg("unsloth")

HAS_CUDA = False
TORCH_RUNTIME_STR = "⏳ Не установлен"
VRAM_GB = float(PRIMARY_NVIDIA_GPU["vram_gb"]) if PRIMARY_NVIDIA_GPU else (
    float(PHYSICAL_GPUS[0]["vram_gb"]) if PHYSICAL_GPUS else 0.0
)

if HAS_TORCH:
    import torch
    HAS_CUDA = bool(torch.cuda.is_available())
    t_ver = getattr(torch, "__version__", "unknown")
    t_cuda = getattr(torch.version, "cuda", None)
    if HAS_CUDA:
        dev_name = torch.cuda.get_device_name(0)
        VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
        TORCH_RUNTIME_STR = f"✅ CUDA активна ({dev_name} | {VRAM_GB} ГБ VRAM | torch {t_ver})"
    elif HAS_NVIDIA_HW and t_cuda is None:
        TORCH_RUNTIME_STR = (
            f"⚠️ Установлен CPU-вариант PyTorch (torch {t_ver} БЕЗ CUDA)! "
            f"Поставьте AUTO_FIX_GPU_PYTORCH = True в 1-й строке этой клетки и нажмите Shift+Enter"
        )
    elif HAS_NVIDIA_HW:
        TORCH_RUNTIME_STR = (
            f"⚠️ torch {t_ver} (CUDA {t_cuda}) не смог подключиться к драйверу NVIDIA — обновите драйвер Game Ready / Studio или перезапустите ядро"
        )
    else:
        TORCH_RUNTIME_STR = f"ℹ️ torch {t_ver} (режим CPU)"
elif HAS_NVIDIA_HW:
    TORCH_RUNTIME_STR = (
        f"⏳ Видеокарта NVIDIA готова, но PyTorch ещё не установлен -> "
        f"поставьте AUTO_FIX_GPU_PYTORCH = True в 1-й строке этой клетки (или используйте Клетку 8)"
    )

# Формируем понятную 2-ю строчку таблицы по всем физическим видеокартам ПК
if PHYSICAL_GPUS:
    main_g = PHYSICAL_GPUS[0]
    vram_part = f"{main_g['vram_gb']} ГБ VRAM" if main_g["vram_gb"] > 0 else "объём VRAM уточняется драйвером"
    cuda_part = f" | макс. CUDA {main_g['driver_cuda']}" if main_g.get("driver_cuda") else ""
    drv_part = f" | драйвер {main_g['driver']}" if main_g.get("driver") else ""
    extra_gpus = f" (+ доп. адаптер: {', '.join(g['name'] for g in PHYSICAL_GPUS[1:])})" if len(PHYSICAL_GPUS) > 1 else ""
    GPU_INFO = f"✅ {main_g['name']} ({vram_part}{drv_part}{cuda_part}){extra_gpus}"
elif HAS_CUDA:
    GPU_INFO = TORCH_RUNTIME_STR
else:
    GPU_INFO = "Не обнаружена (доступен CPU-режим + экспорт в Google Colab GPU)"

hw_table = pd.DataFrame([
    {"Компонент": "1. Операционная система / Python", "Статус": f"{platform.system()} {platform.release()} | Python {platform.python_version()}"},
    {"Компонент": "2. Видеокарта ПК (физическое устройство)", "Статус": GPU_INFO},
    {"Компонент": "3. Связка PyTorch <-> GPU (CUDA)", "Статус": TORCH_RUNTIME_STR},
    {"Компонент": "4. Табличный ML (scikit-learn)", "Статус": "✅ Установлен" if HAS_SKLEARN else "ℹ️ Включён встроенный NumPy ML-движок"},
    {"Компонент": "5. Transformers + PEFT + TRL (QLoRA)", "Статус": "✅ Готовы" if (HAS_TRANSFORMERS and HAS_PEFT and HAS_TRL) else "⏳ Не установлены (ставятся в Клетке 8)"},
    {"Компонент": "6. Unsloth (2x ускорение QLoRA)", "Статус": "✅ Доступен" if HAS_UNSLOTH else "ℹ️ Опционально (используется стандартный PEFT/TRL)"},
])
display(hw_table)

if HAS_NVIDIA_HW and not HAS_CUDA:
    py_exe = sys.executable
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    bat_path = PROJECT_ROOT / "install_gpu_pytorch.bat"
    if platform.system() == "Windows":
        bat_lines = [
            "@echo off",
            "chcp 65001 >nul",
            "echo ======================================================================",
            f"echo Установка PyTorch CUDA ({DETECTED_CUDA_WHEEL_TAG}) + QLoRA для {PRIMARY_NVIDIA_GPU['name']}",
            f"echo Целевой Python ноутбука: \"{py_exe}\"",
            "echo ВАЖНО: Сначала ЗАКРОЙТЕ окно Jupyter / VS Code, чтобы разблокировать DLL!",
            "echo ======================================================================",
            "pause",
            f"\"{py_exe}\" -m pip uninstall -y torch torchvision torchaudio",
            f"\"{py_exe}\" -m pip install torch --index-url https://download.pytorch.org/whl/{DETECTED_CUDA_WHEEL_TAG}",
            f"\"{py_exe}\" -m pip install -U \"transformers>=4.57.0\" \"peft>=0.15.0\" \"trl>=0.16.0\" \"datasets>=3.2.0\" \"accelerate>=1.4.0\" \"bitsandbytes>=0.45.0\"",
            "echo.",
            "echo === ПРОВЕРКА ПОДКЛЮЧЕНИЯ ВИДЕОКАРТЫ ===",
            f"\"{py_exe}\" -c \"import torch; print('CUDA OK:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO')\"",
            "pause",
        ]
        bat_path.write_text("\r\n".join(bat_lines) + "\r\n", encoding="utf-8")

    msg_lines = [
        f"💡 Причина найдена: ваша видеокарта «{PRIMARY_NVIDIA_GPU['name']}» видна системе, но в этом Python ({py_tag}) не стоит CUDA-сборка PyTorch.",
        f"🛠️ В папке проекта автоматически создан готовый установщик: {bat_path.resolve()}",
        "   1) Закройте ноутбук / VS Code (иначе Windows не даст заменить открытые .dll файлы старого torch).",
        "   2) Дважды кликните по файлу install_gpu_pytorch.bat — он сам поставит нужный CUDA PyTorch именно в ваш Python!",
        f'   (Или выполните вручную в CMD: "{py_exe}" -m pip uninstall -y torch torchvision torchaudio && "{py_exe}" -m pip install torch --index-url https://download.pytorch.org/whl/{DETECTED_CUDA_WHEEL_TAG})',
    ]
    print("\n".join(msg_lines))
elif PHYSICAL_GPUS and not HAS_NVIDIA_HW:
    print(
        f"ℹ️ Обнаружена видеокарта «{PHYSICAL_GPUS[0]['name']}» ({PHYSICAL_GPUS[0]['vendor']}). "
        "4-битное QLoRA-обучение (bitsandbytes) использует ядра NVIDIA CUDA: разметку Учителем (Клетки 3–6) и Табличный ИИ (Клетка 7) "
        "выполняйте прямо здесь на вашем ПК, а для QLoRA в Клетке 10 за 1 секунду соберётся готовый архив под бесплатный GPU Google Colab!"
    )
elif HAS_CUDA:
    rec_model = "Qwen/Qwen3.5-9B или Qwen/Qwen3.5-4B" if VRAM_GB >= 12.0 else "Qwen/Qwen3.5-4B"
    print(f"🚀 Видеокарта полностью подключена к CUDA ({VRAM_GB} ГБ VRAM)! Рекомендуемая локальная модель: {rec_model}")


## Клетка 3. Сборка компактных паспортов `facts_<город>.json` из `data_cleaned` и `artifacts_epanet_ai`
Читает рассчитанные вами на прошлом этапе таблицы (`Узел_этажность_и_свободный_напор.csv`, `Труба_скорости_и_режимы_Walski.csv`, `Насос_рабочие_точки_и_мощность.csv`, `График_потребления_анализ_24ч.csv`, `pruned_zero_manifest.json` и `summaries/`) и формирует **строгий JSON-паспорт каждого города** — именно этот структурированный блок данных передаётся на вход ИИ!

In [ ]:
import io


def safe_read_csv(fpath: Path) -> pd.DataFrame:
    """Надёжное чтение CSV для Windows: обходит баг C-движка Pandas с кириллическими путями на дисках D:/E: и не падает на пустых файлах."""
    if not fpath.exists():
        return pd.DataFrame()
    try:
        if fpath.stat().st_size == 0:
            return pd.DataFrame()
    except OSError:
        return pd.DataFrame()

    raw_text: str | None = None
    for enc in ("utf-8-sig", "utf-8", "cp1251"):
        try:
            with fpath.open("r", encoding=enc, errors="strict") as fh:
                raw_text = fh.read()
            break
        except UnicodeDecodeError:
            continue
        except OSError:
            return pd.DataFrame()

    if raw_text is None:
        raw_text = fpath.read_text(encoding="utf-8", errors="replace")

    stripped = raw_text.lstrip("\ufeff").strip()
    if not stripped:
        return pd.DataFrame()

    try:
        return pd.read_csv(io.StringIO(stripped))
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def resolve_subdir_unicode(parent: Path, target_name: str) -> Path:
    """Находит подпапку с учётом Unicode-нормализации NFC/NFD и регистра кириллицы."""
    direct = parent / target_name
    if direct.exists():
        return direct
    if not parent.exists():
        return direct
    norm_t = unicodedata.normalize("NFC", target_name).strip().lower()
    for sub in parent.iterdir():
        if sub.is_dir() and unicodedata.normalize("NFC", sub.name).strip().lower() == norm_t:
            return sub
    return direct


def discover_cleaned_cities(cleaned_root: Path) -> list[Path]:
    """Находит все обработанные папки городов внутри data_cleaned/."""
    if not cleaned_root.exists():
        return []
    if (cleaned_root / "расчетные_параметры").exists() or (cleaned_root / "Узел.csv").exists():
        return [cleaned_root]
    cities = []
    for sub in sorted(cleaned_root.iterdir(), key=lambda p: unicodedata.normalize("NFC", p.name).lower()):
        if sub.is_dir() and not sub.name.startswith("."):
            if (sub / "расчетные_параметры").exists() or any(p.suffix.lower() == ".csv" for p in sub.iterdir() if p.is_file()):
                cities.append(sub)
    return cities


def safe_float(val: Any, decimals: int = 3) -> float | None:
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return None
    try:
        f = float(val)
        return None if math.isnan(f) else round(f, decimals)
    except (ValueError, TypeError):
        return None


def _reconstruct_nodes_when_req_pressure_zero(city_name: str, city_dir: Path, summary_dir: Path, top_k: int) -> tuple[dict[str, Any], str]:
    """Если в исходном Узел.csv у всех узлов было Требуемое давление = 0 (из-за чего Узел_этажность... пуст),
    восстанавливает реальное водопотребление и свободные напоры HGL - Z по Потребление Узел, Напор Узел и data/<город>/Узел.csv!"""
    s_dem = safe_read_csv(summary_dir / "summary_node_demand.csv")
    s_head = safe_read_csv(summary_dir / "summary_node_head.csv")

    consumer_cnt = 0
    daily_m3 = 0.0
    peak_q = 0.0
    consumer_ids: set[str] = set()

    if not s_dem.empty and {"id", "mean_val", "max_val"}.issubset(s_dem.columns):
        consumer_cnt = int(len(s_dem))
        daily_m3 = safe_float(float(s_dem["mean_val"].fillna(0.0).sum()) * 24.0, 2) or 0.0
        peak_q = safe_float(float(s_dem["max_val"].fillna(0.0).sum()), 2) or 0.0
        consumer_ids = set(s_dem["id"].astype(str))
    else:
        f_cons = city_dir / "Потребители Узел.csv"
        cdf = safe_read_csv(f_cons)
        if not cdf.empty:
            id_c = cdf.columns[0]
            q_c = next((c for c in cdf.columns if "потреблен" in c.lower()), cdf.columns[1] if len(cdf.columns) > 1 else id_c)
            consumer_ids = set(cdf[id_c].astype(str))
            consumer_cnt = len(consumer_ids)
            q_sum = float(pd.to_numeric(cdf[q_c], errors="coerce").fillna(0.0).sum())
            peak_q = safe_float(q_sum, 2) or 0.0
            daily_m3 = safe_float(q_sum * 24.0, 2) or 0.0

    # Пытаемся подтянуть высотные отметки Z из исходной папки data/<город>/Узел.csv (если она рядом)
    elev_map: dict[str, float] = {}
    raw_city_dir = resolve_subdir_unicode(PROJECT_ROOT / "data", city_name)
    raw_nodes_df = safe_read_csv(raw_city_dir / "Узел.csv")
    if not raw_nodes_df.empty and len(raw_nodes_df.columns) >= 2:
        id_col = raw_nodes_df.columns[0]
        z_col = next((c for c in raw_nodes_df.columns if "высот" in c.lower() or "отмет" in c.lower()), raw_nodes_df.columns[1])
        for _, r in raw_nodes_df.iterrows():
            zv = safe_float(r[z_col], 3)
            if zv is not None:
                elev_map[str(r[id_col]).strip()] = zv

    free_min_net: float | None = None
    free_max_net: float | None = None
    excess_items: list[dict[str, Any]] = []
    excess_cnt = 0

    if not s_head.empty and {"id", "min_val", "max_val"}.issubset(s_head.columns) and elev_map:
        f_mins: list[float] = []
        f_maxs: list[tuple[str, float]] = []
        for _, r in s_head.iterrows():
            nid = str(r["id"]).strip()
            if consumer_ids and nid not in consumer_ids:
                continue
            if nid in elev_map:
                h_mn = safe_float(r["min_val"], 3)
                h_mx = safe_float(r["max_val"], 3)
                if h_mn is not None:
                    f_mins.append(round(h_mn - elev_map[nid], 2))
                if h_mx is not None:
                    f_maxs.append((nid, round(h_mx - elev_map[nid], 2)))
        if f_mins:
            free_min_net = round(min(f_mins), 2)
        if f_maxs:
            free_max_net = round(max(x[1] for x in f_maxs), 2)
            over_60 = sorted([x for x in f_maxs if x[1] > 60.0], key=lambda t: t[1], reverse=True)
            excess_cnt = len(over_60)
            excess_items = [
                {
                    "node_id": nid,
                    "floors": 0,
                    "free_pressure_max_m": fmx,
                    "pressure_max_bar": round(fmx * 0.0980665, 2),
                    "source": {"file": "summary_node_head.csv + data/Узел.csv", "aggregation": "HGL_max - Z"},
                }
                for nid, fmx in over_60[:top_k]
            ]

    flag_note = (
        f"В исходной таблице Узел.csv города «{city_name}» на всех узлах стояло Требуемое давление = 0 "
        "(нормативная этажность не задавалась автором модели); показатели водоразбора и напоров собраны по фактическим узлам-абонентам"
    )
    summary_dict = {
        "consumer_nodes_count": consumer_cnt,
        "total_daily_consumption_m3_day": daily_m3,
        "sum_peak_node_demand_m3_h": peak_q,
        "free_pressure_network_min_m": free_min_net,
        "free_pressure_network_max_m": free_max_net,
        "deficit_nodes_count": 0,
        "excess_pressure_over_60m_nodes_count": excess_cnt,
        "floors_distribution": {},
        "worst_deficit_nodes": [],
        "highest_excess_pressure_nodes": excess_items,
    }
    return summary_dict, flag_note


def extract_city_facts_json(city_dir: Path, artifacts_root: Path, top_k: int = 8) -> dict[str, Any]:
    """Собирает детерминированный пакет фактов facts.json со строгой трассировкой источников."""
    city_name = unicodedata.normalize("NFC", city_dir.name)
    derived_dir = resolve_subdir_unicode(city_dir, "расчетные_параметры")
    summary_dir = resolve_subdir_unicode(artifacts_root / "summaries", city_name)

    pruned_manifest = {}
    pm_path = city_dir / "pruned_zero_manifest.json"
    if pm_path.exists():
        try:
            pruned_manifest = json.loads(pm_path.read_text(encoding="utf-8"))
        except Exception:
            pruned_manifest = {}

    facts: dict[str, Any] = {
        "city_name": city_name,
        "schema_version": "epanet_ai_facts_v1",
        "traceability_note": "Все числа извлечены детерминированно из data_cleaned/<город>/расчетные_параметры/ без пересчёта в LLM",
        "zero_cleaning_summary": {
            "pruned_transit_nodes_count": len(pruned_manifest.get("pruned_transit_node_ids", [])),
            "pruned_standby_pumps_count": len(pruned_manifest.get("pruned_standby_pump_ids", [])),
            "pruned_zero_consumers_count": int(pruned_manifest.get("pruned_zero_consumers_count", 0)),
        },
        "nodes_summary": {
            "consumer_nodes_count": 0,
            "total_daily_consumption_m3_day": 0.0,
            "sum_peak_node_demand_m3_h": 0.0,
            "free_pressure_network_min_m": None,
            "free_pressure_network_max_m": None,
            "deficit_nodes_count": 0,
            "excess_pressure_over_60m_nodes_count": 0,
            "floors_distribution": {},
            "worst_deficit_nodes": [],
            "highest_excess_pressure_nodes": [],
        },
        "pipes_summary": {
            "pipes_count": 0,
            "total_length_km": 0.0,
            "total_internal_water_volume_m3": 0.0,
            "network_max_velocity_m_s": None,
            "pipes_over_1_5_m_s_count": 0,
            "pipes_over_3_0_m_s_water_hammer_risk_count": 0,
            "stagnant_pipes_under_0_3_m_s_count": 0,
            "reverse_flow_pipes_count": 0,
            "self_loop_topology_pipes_count": 0,
            "top_high_velocity_pipes": [],
            "top_stagnant_pipes": [],
            "top_reverse_flow_pipes": [],
        },
        "pumps_summary": {
            "total_pumps_declared": 0,
            "active_pumps_count": 0,
            "standby_off_pumps_count": 0,
            "total_pumped_volume_m3_day": 0.0,
            "total_hydraulic_energy_kwh_day": 0.0,
            "active_pumps_details": [],
        },
        "schedules_summary": {
            "schedules_count": 0,
            "all_24_hours_valid": True,
            "max_peaking_factor_k_max": None,
            "min_night_factor_k_min": None,
        },
        "data_quality_flags": [],
    }

    # 1. Факты по узлам, этажности и свободному напору
    f_nodes = derived_dir / "Узел_этажность_и_свободный_напор.csv"
    ndf = safe_read_csv(f_nodes)
    if not ndf.empty and "ID" in ndf.columns:
        total_nodes = len(ndf)
        total_daily_m3 = safe_float(ndf["Суточный_объем_м3_сут"].sum(), 2) if "Суточный_объем_м3_сут" in ndf.columns else 0.0
        peak_q_sum = safe_float(ndf["Пиковый_расход_м3_ч"].sum(), 2) if "Пиковый_расход_м3_ч" in ndf.columns else 0.0

        fl_counts = (
            ndf[ndf["Этажность_дома"] > 0]["Этажность_дома"]
            .value_counts()
            .sort_index()
            .to_dict()
            if "Этажность_дома" in ndf.columns else {}
        )
        floors_breakdown = {f"{int(k)}_эт": int(v) for k, v in fl_counts.items()}

        has_margin = ndf.dropna(subset=["Запас_напора_мин_м"]).copy() if "Запас_напора_мин_м" in ndf.columns else pd.DataFrame()
        deficit_df = has_margin[has_margin["Запас_напора_мин_м"] < 0.0].sort_values("Запас_напора_мин_м") if not has_margin.empty else pd.DataFrame()
        excess_df = ndf.dropna(subset=["Свободный_напор_макс_м"]) if "Свободный_напор_макс_м" in ndf.columns else pd.DataFrame()
        excess_df = excess_df[excess_df["Свободный_напор_макс_м"] > 60.0].sort_values("Свободный_напор_макс_м", ascending=False) if not excess_df.empty else pd.DataFrame()

        def_min_col = "Минут_дефицита_в_сутки" if "Минут_дефицита_в_сутки" in ndf.columns else "Длительность_дефицита_мин_сут"
        bar_col = "Давление_макс_бар" if "Давление_макс_бар" in ndf.columns else "Макс_давление_бар"

        deficit_items = [
            {
                "node_id": str(r["ID"]),
                "floors": int(r.get("Этажность_дома", 0) or 0),
                "required_pressure_m": safe_float(r.get("Требуемое_давление_м"), 2),
                "free_pressure_min_m": safe_float(r.get("Свободный_напор_мин_м"), 2),
                "margin_min_m": safe_float(r.get("Запас_напора_мин_м"), 2),
                "deficit_minutes_per_day": int(r.get(def_min_col, 0) or 0),
                "daily_demand_m3": safe_float(r.get("Суточный_объем_м3_сут"), 2),
                "source": {"file": f_nodes.name, "aggregation": "row_exact"},
            }
            for _, r in deficit_df.head(top_k).iterrows()
        ]

        excess_items = [
            {
                "node_id": str(r["ID"]),
                "floors": int(r.get("Этажность_дома", 0) or 0),
                "free_pressure_max_m": safe_float(r.get("Свободный_напор_макс_м"), 2),
                "pressure_max_bar": safe_float(r.get(bar_col), 2),
                "source": {"file": f_nodes.name, "aggregation": "row_exact"},
            }
            for _, r in excess_df.head(top_k).iterrows()
        ]

        min_free_overall = safe_float(ndf["Свободный_напор_мин_м"].min(), 2) if "Свободный_напор_мин_м" in ndf.columns else None
        max_free_overall = safe_float(ndf["Свободный_напор_макс_м"].max(), 2) if "Свободный_напор_макс_м" in ndf.columns else None

        facts["nodes_summary"] = {
            "consumer_nodes_count": int(total_nodes),
            "total_daily_consumption_m3_day": total_daily_m3 or 0.0,
            "sum_peak_node_demand_m3_h": peak_q_sum or 0.0,
            "free_pressure_network_min_m": min_free_overall,
            "free_pressure_network_max_m": max_free_overall,
            "deficit_nodes_count": int(len(deficit_df)),
            "excess_pressure_over_60m_nodes_count": int(len(excess_df)),
            "floors_distribution": floors_breakdown,
            "worst_deficit_nodes": deficit_items,
            "highest_excess_pressure_nodes": excess_items,
        }
    else:
        fb_nodes, fb_flag = _reconstruct_nodes_when_req_pressure_zero(city_name, city_dir, summary_dir, top_k)
        facts["nodes_summary"] = fb_nodes
        facts["data_quality_flags"].append(fb_flag)

    # 2. Факты по трубам, скоростям (Walski Eq. 2.9), числу Рейнольдса и реверсам потока
    f_pipes = derived_dir / "Труба_скорости_и_режимы_Walski.csv"
    pdf = safe_read_csv(f_pipes)
    if not pdf.empty and "ID" in pdf.columns:
        re_col = "Число_Рейнольдса_Re" if "Число_Рейнольдса_Re" in pdf.columns else "Число_Рейнольдса_Re_макс"
        q_max_col = "Расход_макс_м3_ч" if "Расход_макс_м3_ч" in pdf.columns else "Расход_макс_абс_м3_ч"
        qd_col = "Индекс_Поток_делить_Диаметр" if "Индекс_Поток_делить_Диаметр" in pdf.columns else "Упрощенный_индекс_Qмакс_делить_D"

        total_length_km = safe_float(pdf["Длина_м"].sum() / 1000.0, 3) if "Длина_м" in pdf.columns else 0.0
        total_pipe_vol_m3 = safe_float(pdf["Объем_трубы_м3"].sum(), 2) if "Объем_трубы_м3" in pdf.columns else 0.0

        self_loops_n = int((pdf["Начало"].astype(str) == pdf["Конец"].astype(str)).sum()) if {"Начало", "Конец"}.issubset(pdf.columns) else 0
        if self_loops_n > 0:
            facts["data_quality_flags"].append(
                f"В Труба.csv обнаружено {self_loops_n} участков с петлёй экспорта Начало == Конец (потери напора dH по ним заблокированы до исправления топологии в AquaBit)"
            )

        v_valid = pdf.dropna(subset=["Скорость_макс_м_с"]).copy() if "Скорость_макс_м_с" in pdf.columns else pd.DataFrame()
        high_v_df = v_valid[v_valid["Скорость_макс_м_с"] > 1.5].sort_values("Скорость_макс_м_с", ascending=False) if not v_valid.empty else pd.DataFrame()
        over_3ms_count = int((v_valid["Скорость_макс_м_с"] > 3.0).sum()) if not v_valid.empty else 0
        stagnant_df = (
            v_valid[v_valid["Скорость_макс_м_с"] < 0.30].sort_values("Время_пробега_воды_мин", ascending=False, na_position="last")
            if (not v_valid.empty and "Время_пробега_воды_мин" in v_valid.columns) else pd.DataFrame()
        )
        reverse_df = (
            pdf[pdf["Шагов_обратного_потока"] > 0].sort_values("Шагов_обратного_потока", ascending=False)
            if "Шагов_обратного_потока" in pdf.columns else pd.DataFrame()
        )

        reversals_map: dict[str, int] = {}
        spdf = safe_read_csv(summary_dir / "summary_pipe_flow.csv")
        if not spdf.empty and {"id", "flow_reversals_count"}.issubset(spdf.columns):
            reversals_map = dict(zip(spdf["id"].astype(str), spdf["flow_reversals_count"].fillna(0).astype(int)))

        facts["pipes_summary"] = {
            "pipes_count": int(len(pdf)),
            "total_length_km": total_length_km or 0.0,
            "total_internal_water_volume_m3": total_pipe_vol_m3 or 0.0,
            "network_max_velocity_m_s": safe_float(v_valid["Скорость_макс_м_с"].max(), 3) if not v_valid.empty else None,
            "pipes_over_1_5_m_s_count": int(len(high_v_df)),
            "pipes_over_3_0_m_s_water_hammer_risk_count": over_3ms_count,
            "stagnant_pipes_under_0_3_m_s_count": int(len(stagnant_df)),
            "reverse_flow_pipes_count": int(len(reverse_df)),
            "self_loop_topology_pipes_count": self_loops_n,
            "top_high_velocity_pipes": [
                {
                    "pipe_id": str(r["ID"]),
                    "diameter_mm": safe_float(r.get("Диаметр_мм"), 0),
                    "length_m": safe_float(r.get("Длина_м"), 1),
                    "q_max_m3_h": safe_float(r.get(q_max_col), 2),
                    "q_over_d_index": safe_float(r.get(qd_col), 4),
                    "velocity_max_m_s": safe_float(r.get("Скорость_макс_м_с"), 3),
                    "reynolds_max": safe_float(r.get(re_col), 0),
                    "source": {"file": f_pipes.name, "aggregation": "row_exact"},
                }
                for _, r in high_v_df.head(top_k).iterrows()
            ],
            "top_stagnant_pipes": [
                {
                    "pipe_id": str(r["ID"]),
                    "diameter_mm": safe_float(r.get("Диаметр_мм"), 0),
                    "length_m": safe_float(r.get("Длина_м"), 1),
                    "velocity_max_m_s": safe_float(r.get("Скорость_макс_м_с"), 4),
                    "travel_time_min": safe_float(r.get("Время_пробега_воды_мин"), 1),
                    "source": {"file": f_pipes.name, "aggregation": "row_exact"},
                }
                for _, r in stagnant_df.head(top_k).iterrows()
            ],
            "top_reverse_flow_pipes": [
                {
                    "pipe_id": str(r["ID"]),
                    "reverse_steps_of_1440": int(r.get("Шагов_обратного_потока", 0) or 0),
                    "direction_sign_flips_per_day": int(reversals_map.get(str(r["ID"]), 0)),
                    "velocity_max_m_s": safe_float(r.get("Скорость_макс_м_с"), 3),
                }
                for _, r in reverse_df.head(top_k).iterrows()
            ],
        }
    else:
        # Автовосстановление сводки по трубам для сверхкрупных городов (например, Алматы 148 тыс. узлов),
        # если расчетные_параметры/Труба_скорости... ещё не сгенерирована, но есть Труба.csv + summary_pipe_flow.csv!
        raw_pipes_df = safe_read_csv(city_dir / "Труба.csv")
        if raw_pipes_df.empty:
            raw_pipes_df = safe_read_csv(resolve_subdir_unicode(PROJECT_ROOT / "data", city_name) / "Труба.csv")
        spdf = safe_read_csv(summary_dir / "summary_pipe_flow.csv")
        if not raw_pipes_df.empty and len(raw_pipes_df.columns) >= 5:
            p_id_c = raw_pipes_df.columns[0]
            s_c = next((c for c in raw_pipes_df.columns if "начал" in c.lower()), raw_pipes_df.columns[1])
            e_c = next((c for c in raw_pipes_df.columns if "конец" in c.lower()), raw_pipes_df.columns[2])
            d_c = next((c for c in raw_pipes_df.columns if "диаметр" in c.lower()), raw_pipes_df.columns[3])
            l_c = next((c for c in raw_pipes_df.columns if "длин" in c.lower()), raw_pipes_df.columns[4])

            d_m = pd.to_numeric(raw_pipes_df[d_c], errors="coerce").fillna(0.0) / 1000.0
            l_m = pd.to_numeric(raw_pipes_df[l_c], errors="coerce").fillna(0.0)
            area_m2 = np.pi * (d_m ** 2) / 4.0
            vol_m3 = float((area_m2 * l_m).sum())
            len_km = float(l_m.sum() / 1000.0)
            sloops = int((raw_pipes_df[s_c].astype(str) == raw_pipes_df[e_c].astype(str)).sum())

            net_vmax = None
            over_15 = over_30 = stag_cnt = rev_cnt = 0
            if not spdf.empty and {"id", "min_val", "max_val", "negative_steps_count"}.issubset(spdf.columns):
                merged_p = raw_pipes_df[[p_id_c, d_c, l_c]].copy()
                merged_p["id_str"] = merged_p[p_id_c].astype(str)
                sp_copy = spdf.copy()
                sp_copy["id_str"] = sp_copy["id"].astype(str)
                m_df = merged_p.merge(sp_copy, on="id_str", how="inner")
                if not m_df.empty:
                    q_abs = np.maximum(
                        pd.to_numeric(m_df["min_val"], errors="coerce").abs().fillna(0.0),
                        pd.to_numeric(m_df["max_val"], errors="coerce").abs().fillna(0.0),
                    )
                    dm_arr = pd.to_numeric(m_df[d_c], errors="coerce").fillna(0.0) / 1000.0
                    with np.errstate(divide="ignore", invalid="ignore"):
                        v_arr = np.where(dm_arr > 0, (4.0 * (q_abs / 3600.0)) / (np.pi * (dm_arr ** 2)), np.nan)
                    valid_v = v_arr[~np.isnan(v_arr)]
                    if valid_v.size > 0:
                        net_vmax = safe_float(float(np.max(valid_v)), 3)
                        over_15 = int(np.sum(valid_v > 1.5))
                        over_30 = int(np.sum(valid_v > 3.0))
                        stag_cnt = int(np.sum(valid_v < 0.30))
                    rev_cnt = int((pd.to_numeric(m_df["negative_steps_count"], errors="coerce").fillna(0) > 0).sum())

            facts["pipes_summary"] = {
                "pipes_count": int(len(raw_pipes_df)),
                "total_length_km": safe_float(len_km, 3) or 0.0,
                "total_internal_water_volume_m3": safe_float(vol_m3, 2) or 0.0,
                "network_max_velocity_m_s": net_vmax,
                "pipes_over_1_5_m_s_count": over_15,
                "pipes_over_3_0_m_s_water_hammer_risk_count": over_30,
                "stagnant_pipes_under_0_3_m_s_count": stag_cnt,
                "reverse_flow_pipes_count": rev_cnt,
                "self_loop_topology_pipes_count": sloops,
                "top_high_velocity_pipes": [],
                "top_stagnant_pipes": [],
                "top_reverse_flow_pipes": [],
            }

    # 3. Факты по насосным агрегатам (безопасно обрабатывает и самотечные сети с 0 насосов!)
    f_pumps = derived_dir / "Насос_рабочие_точки_и_мощность.csv"
    pmdf = safe_read_csv(f_pumps)
    if not pmdf.empty and "ID" in pmdf.columns and "Моточасов_в_сутки" in pmdf.columns:
        starts_col = "Число_пусков" if "Число_пусков" in pmdf.columns else "Число_пусков_за_сутки"
        kw_col = "Гидравл_мощность_сред_кВт" if "Гидравл_мощность_сред_кВт" in pmdf.columns else "Гидравлическая_мощность_сред_кВт"
        kwh_col = "Гидравл_энергия_кВтч_сут" if "Гидравл_энергия_кВтч_сут" in pmdf.columns else "Гидравлическая_энергия_кВтч_сут"
        tdh_col = "Расчетный_напор_TDH_м" if "Расчетный_напор_TDH_м" in pmdf.columns else "Расчетный_напор_TDH_сред_м"
        q_col = "Подача_сред_м3_ч" if "Подача_сред_м3_ч" in pmdf.columns else "Подача_рабочая_сред_м3_ч"

        active_pumps = pmdf[pmdf["Моточасов_в_сутки"].fillna(0.0) > 0.0]
        standby_pumps = pmdf[pmdf["Моточасов_в_сутки"].fillna(0.0) <= 0.0]
        total_energy_kwh = safe_float(pmdf[kwh_col].fillna(0.0).sum(), 2) if kwh_col in pmdf.columns else 0.0
        total_pumped_m3 = safe_float(pmdf["Суточная_подача_м3_сут"].fillna(0.0).sum(), 2) if "Суточная_подача_м3_сут" in pmdf.columns else 0.0

        facts["pumps_summary"] = {
            "total_pumps_declared": int(len(pmdf)),
            "active_pumps_count": int(len(active_pumps)),
            "standby_off_pumps_count": int(len(standby_pumps)),
            "total_pumped_volume_m3_day": total_pumped_m3 or 0.0,
            "total_hydraulic_energy_kwh_day": total_energy_kwh or 0.0,
            "active_pumps_details": [
                {
                    "pump_id": str(r["ID"]),
                    "curve_id": str(r.get("Кривая_QH", "")),
                    "run_hours_per_day": safe_float(r.get("Моточасов_в_сутки"), 2),
                    "starts_per_day": int(r.get(starts_col, 0) or 0),
                    "flow_mean_m3_h": safe_float(r.get(q_col), 2),
                    "flow_max_m3_h": safe_float(r.get("Подача_макс_м3_ч"), 2),
                    "tdh_head_m": safe_float(r.get(tdh_col), 2),
                    "hydraulic_power_kw": safe_float(r.get(kw_col), 2),
                    "hydraulic_energy_kwh_day": safe_float(r.get(kwh_col), 2),
                }
                for _, r in active_pumps.iterrows()
            ],
        }
    else:
        raw_pumps_df = safe_read_csv(city_dir / "Насос.csv")
        sp_pump = safe_read_csv(summary_dir / "summary_pump_flow.csv")
        if not raw_pumps_df.empty:
            tot_decl = int(len(raw_pumps_df))
            act_cnt = int((pd.to_numeric(sp_pump.get("max_val", pd.Series(dtype=float)), errors="coerce").fillna(0.0) > 0).sum()) if not sp_pump.empty else 0
            pumped_day = safe_float(float(pd.to_numeric(sp_pump.get("mean_val", pd.Series(dtype=float)), errors="coerce").fillna(0.0).sum()) * 24.0, 2) if not sp_pump.empty else 0.0
            facts["pumps_summary"] = {
                "total_pumps_declared": tot_decl,
                "active_pumps_count": act_cnt,
                "standby_off_pumps_count": max(0, tot_decl - act_cnt),
                "total_pumped_volume_m3_day": pumped_day or 0.0,
                "total_hydraulic_energy_kwh_day": 0.0,
                "active_pumps_details": [],
            }

    # 4. Факты по суточным графикам потребления
    f_sched = derived_dir / "График_потребления_анализ_24ч.csv"
    sdf = safe_read_csv(f_sched)
    if not sdf.empty:
        rows_col = "Строк_часов" if "Строк_часов" in sdf.columns else "Часовых_строк"
        non_24 = int((sdf[rows_col] != 24).sum()) if rows_col in sdf.columns else 0
        if non_24 > 0:
            facts["data_quality_flags"].append(f"В График потребления.csv найдено {non_24} графиков с числом точек != 24")
        facts["schedules_summary"] = {
            "schedules_count": int(len(sdf)),
            "all_24_hours_valid": bool(non_24 == 0),
            "max_peaking_factor_k_max": safe_float(sdf["K_макс"].max(), 4) if "K_макс" in sdf.columns else None,
            "min_night_factor_k_min": safe_float(sdf["K_мин"].min(), 4) if "K_мин" in sdf.columns else None,
        }

    return facts


CLEANED_CITY_DIRS = discover_cleaned_cities(DATA_CLEANED_DIR)
ALL_CITY_FACTS: dict[str, dict[str, Any]] = {}

if not CLEANED_CITY_DIRS:
    print(f"⚠️ В папке {DATA_CLEANED_DIR.resolve()} пока нет городов. Сначала запустите EPANET_AI_Pipeline.ipynb!")
else:
    overview_rows = []
    for cdir in CLEANED_CITY_DIRS:
        cfacts = extract_city_facts_json(cdir, ARTIFACTS_DIR, top_k=TOP_K_CRITICAL_ITEMS)
        cname = cfacts["city_name"]
        ALL_CITY_FACTS[cname] = cfacts
        out_f = FACTS_DIR / f"facts_{cname}.json"
        out_f.write_text(json.dumps(cfacts, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

        ns = cfacts.get("nodes_summary", {})
        ps = cfacts.get("pipes_summary", {})
        pms = cfacts.get("pumps_summary", {})
        overview_rows.append({
            "Город": cname,
            "Узлов-абонентов": ns.get("consumer_nodes_count", 0),
            "Узлов с дефицитом напора": ns.get("deficit_nodes_count", 0),
            "Узлов >60 м": ns.get("excess_pressure_over_60m_nodes_count", 0),
            "Труб всего": ps.get("pipes_count", 0),
            "Труб <0.3 м/с (застой)": ps.get("stagnant_pipes_under_0_3_m_s_count", 0),
            "Труб с реверсом (Q<0)": ps.get("reverse_flow_pipes_count", 0),
            "Насосов в работе / резерве": f"{pms.get('active_pumps_count', 0)} / {pms.get('standby_off_pumps_count', 0)}",
            "Энергия насосов (кВт·ч/сут)": pms.get("total_hydraulic_energy_kwh_day", 0.0),
            "Размер паспорта JSON (КБ)": round(out_f.stat().st_size / 1024.0, 1),
        })

    print(f"✅ Собрано паспортов городов для обучения ИИ: {len(ALL_CITY_FACTS)} (сохранены в {FACTS_DIR.resolve()})")
    display(pd.DataFrame(overview_rows))


## Клетка 4. Подготовка пула задач для Модели-Учителя (`teacher_prompts_pool.jsonl`) + опорных примеров
Формирует по каждому вашему городу из `ALL_CITY_FACTS`:
1. **Широкий пул неразмеченных инженерных запросов для Модели-Учителя** (`TEACHER_TASK_POOL` — по 9–15 разноплановых реальных сценариев на каждый город: записка главному инженеру, выдача техусловий на подключение новой застройки, борьба с застоем воды и гидроударом по книге Walski, настройка насосов и ЧРП/PRV, ночной режим для диспетчера, аудит топологии, ловушки на отказ выдумывать КПД/хлор/рубли);
2. **Готовые текстовые карточки `training_epanet_ai/web_chat_batches/`** — на случай, если у вас нет платного API-ключа, но есть доступ к сильной модели в обычном браузерном чате;
3. **Базовые эталонные примеры (`SEED_EXAMPLES`)**, чтобы пайплайн обучения работал даже до первого вызова API Учителя.

In [ ]:
SYSTEM_PROMPT_EPANET_AI = (
    "Ты — экспертная инженерная ИИ-система интерпретации гидравлических расчётов программного комплекса «Аквабит» (AquaBit) "
    "для водопроводных сетей (с опорой на критерии СП 31.13330.2021 и справочника Walski et al. 'Advanced Water Distribution Modeling and Management'). "
    "Всегда упоминай в ответе расчётный комплекс «Аквабит» (AquaBit) и его таблицы (Узел.csv, Труба.csv, Насос.csv, Кривая QH.csv, Потребители Узел.csv) — "
    "НЕ называй модель файлом *.inp и НЕ подменяй название ПК «Аквабит» на классический EPANET. "
    "Помни: гидравлический расчёт уже выполнен решателем ПК «Аквабит» на полной связной модели сети, а показатели pruned_* в zero_cleaning_summary "
    "означают лишь фильтрацию транзитных узлов (где Требуемое давление = 0) и выключенных резервных насосов при подготовке аналитической сводки (топологическая связность расчёта Аквабит при этом полностью сохранена). "
    "Отвечай строго по числам из переданного JSON (facts_inline): запрещено выдумывать или пересчитывать в уме любые цифры, которых нет во входном JSON."
)


def build_training_examples_for_city(facts: dict[str, Any]) -> list[dict[str, Any]]:
    """Создаёт разнообразные обучающие диалоги SFT по одному городу со строгим соблюдением всех чисел из facts."""
    cname = facts["city_name"]
    ns = facts.get("nodes_summary", {})
    ps = facts.get("pipes_summary", {})
    pms = facts.get("pumps_summary", {})
    zs = facts.get("zero_cleaning_summary", {})
    qflags = facts.get("data_quality_flags", [])

    examples: list[dict[str, Any]] = []

    def add_ex(task_type: str, question: str, payload: dict[str, Any], answer_lines: list[str]) -> None:
        user_content = (
            f"Вопрос инженера: {question}\n\n"
            f"Факты расчёта (facts_inline):\n```json\n{json.dumps(payload, ensure_ascii=False, indent=2)}\n```"
        )
        examples.append({
            "city": cname,
            "task_type": task_type,
            "facts_inline": payload,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT_EPANET_AI},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": "\n".join(answer_lines).strip()},
            ],
        })

    # --- 1. Сводное диагностическое заключение по городу ---
    compact_overview = {
        "city_name": cname,
        "zero_cleaning_summary": zs,
        "nodes_overview": {k: v for k, v in ns.items() if not isinstance(v, list)},
        "pipes_overview": {k: v for k, v in ps.items() if not isinstance(v, list)},
        "pumps_overview": {k: v for k, v in pms.items() if not isinstance(v, list)},
        "data_quality_flags": qflags,
    }
    ans_1 = [
        f"### Сводное инженерное заключение по расчётной модели «{cname}»",
        "",
        f"1. **Структура абонентской сети и фильтрация нулей:** в расчёте задействовано **{ns.get('consumer_nodes_count', 0)}** конечных узлов-потребителей "
        f"(при предварительной очистке отфильтровано **{zs.get('pruned_transit_nodes_count', 0)}** промежуточных узлов без требуемого давления и "
        f"**{zs.get('pruned_standby_pumps_count', 0)}** полностью выключенных резервных насосов). "
        f"Суточный объём водопотребления составляет **{ns.get('total_daily_consumption_m3_day', 0.0)} м³/сут**.",
        (f"2. **Режим свободных напоров:** по сети свободный напор меняется в диапазоне от **{ns.get('free_pressure_network_min_m')} м** до **{ns.get('free_pressure_network_max_m')} м**. " if ns.get("free_pressure_network_min_m") is not None else "2. **Режим свободных напоров:** нормативное требуемое давление этажности в таблице Узел.csv не задано (=0). ") +
        f"Дефицит напора ниже требуемого норматива этажности зафиксирован на **{ns.get('deficit_nodes_count', 0)}** узлах; "
        f"избыточный напор свыше 60 м (риск повышенных утечек и износа арматуры) наблюдается на **{ns.get('excess_pressure_over_60m_nodes_count', 0)}** узлах.",
        f"3. **Трубопроводная сеть ({ps.get('pipes_count', 0)} участков, {ps.get('total_length_km', 0.0)} км, внутренний объём {ps.get('total_internal_water_volume_m3', 0.0)} м³):** " +
        (f"максимальная скорость потока достигает **{ps.get('network_max_velocity_m_s')} м/с**. " if ps.get("network_max_velocity_m_s") is not None else "ряды скоростей потока по участкам отсутствуют. ") +
        f"Участков со скоростью свыше 3.0 м/с (риск гидроудара по Walski) — **{ps.get('pipes_over_3_0_m_s_water_hammer_risk_count', 0)}**, "
        f"малоскоростных участков менее 0.3 м/с (риск застоя воды) — **{ps.get('stagnant_pipes_under_0_3_m_s_count', 0)}**, "
        f"труб с обратным направлением потока — **{ps.get('reverse_flow_pipes_count', 0)}**.",
        f"4. **Насосное оборудование:** из **{pms.get('total_pumps_declared', 0)}** агрегатов в работе находятся **{pms.get('active_pumps_count', 0)}**, "
        f"суточная подача — **{pms.get('total_pumped_volume_m3_day', 0.0)} м³/сут**, полезная гидравлическая энергия — **{pms.get('total_hydraulic_energy_kwh_day', 0.0)} кВт·ч/сут**.",
    ]
    if qflags:
        ans_1.append("5. **Замечания к качеству исходных данных модели:** " + "; ".join(qflags) + ".")
    add_ex(
        "city_executive_summary",
        f"Составь сводное инженерное заключение по гидравлическому режиму водопроводной сети города «{cname}».",
        compact_overview,
        ans_1,
    )

    # --- 2. Анализ обеспеченности напором с учётом этажности застройки ---
    nodes_payload = {"city_name": cname, "nodes_summary": ns}
    def_cnt = ns.get("deficit_nodes_count", 0)
    exc_cnt = ns.get("excess_pressure_over_60m_nodes_count", 0)
    ans_2 = [
        f"### Оценка свободных напоров и этажности застройки («{cname}»)",
        "",
        (f"- Всего проанализировано **{ns.get('consumer_nodes_count', 0)}** узлов с абонентской нагрузкой (минимальный свободный напор по сети **{ns.get('free_pressure_network_min_m')} м**, максимальный **{ns.get('free_pressure_network_max_m')} м**)." if ns.get("free_pressure_network_min_m") is not None else f"- Всего проанализировано **{ns.get('consumer_nodes_count', 0)}** узлов с абонентской нагрузкой (требуемое давление этажности в Узел.csv было равно 0)."),
    ]
    if def_cnt == 0:
        ans_2.append(
            "- **Дефицит напора отсутствует (`0` узлов):** на всех конечных узлах свободный напор `HGL - Z` выдерживает норматив СП 31.13330.2021 (`10 м` для 1 этажа + `4 м` на каждый последующий этаж)."
        )
    else:
        ans_2.append(f"- **Обнаружен дефицит свободного напора на `{def_cnt}` узлах.** Наиболее критичные узлы:")
        for item in ns.get("worst_deficit_nodes", [])[:5]:
            ans_2.append(
                f"  - Узел `{item['node_id']}` (застройка **{item['floors']} эт.**, требуется **{item['required_pressure_m']} м**): "
                f"фактический минимум составил **{item['free_pressure_min_m']} м** (нехватка **{item['margin_min_m']} м**, длительность дефицита **{item['deficit_minutes_per_day']} мин/сут**)."
            )
    if exc_cnt > 0:
        ans_2.append(f"- **Зоны избыточного давления (> 60 м):** выявлено **{exc_cnt}** узлов, требующих проверки зонирования или настройки регуляторов давления `PRV`:")
        for item in ns.get("highest_excess_pressure_nodes", [])[:4]:
            ans_2.append(
                f"  - Узел `{item['node_id']}` ({item['floors']} эт.): максимальный свободный напор **{item['free_pressure_max_m']} м** (**{item['pressure_max_bar']} бар**)."
            )
    add_ex(
        "pressure_and_storeys_audit",
        f"Проанализируй обеспеченность свободным напором узлов города «{cname}» с учётом этажности домов и выдели проблемные зоны.",
        nodes_payload,
        ans_2,
    )

    # --- 3. Диагностика скоростей в трубах, застоя воды и реверсов потока ---
    pipes_payload = {"city_name": cname, "pipes_summary": ps}
    ans_3 = [
        f"### Гидравлический анализ трубопроводов («{cname}») по критериям Walski",
        "",
        f"- Проверено **{ps.get('pipes_count', 0)}** участков общей протяжённостью **{ps.get('total_length_km', 0.0)} км** (объём воды в трубах **{ps.get('total_internal_water_volume_m3', 0.0)} м³**).",
        (f"- Максимальная скорость по сети составляет **{ps.get('network_max_velocity_m_s')} м/с**; " if ps.get("network_max_velocity_m_s") is not None else "- ") + f"участков со скоростью свыше `1.5 м/с` — **{ps.get('pipes_over_1_5_m_s_count', 0)}**, свыше `3.0 м/с` (зона риска гидравлического удара) — **{ps.get('pipes_over_3_0_m_s_water_hammer_risk_count', 0)}**.",
    ]
    for p_item in ps.get("top_high_velocity_pipes", [])[:4]:
        ans_3.append(
            f"  - Труба `{p_item['pipe_id']}` (диаметр **{p_item['diameter_mm']} мм**, длина **{p_item['length_m']} м**): "
            f"максимальный расход **{p_item['q_max_m3_h']} м³/ч**, скорость **{p_item['velocity_max_m_s']} м/с** (индекс `Q/D` = **{p_item['q_over_d_index']}**, число Рейнольдса `Re` = **{p_item['reynolds_max']}**)."
        )
    ans_3.append(
        f"- **Малоскоростные и застойные зоны (< 0.3 м/с):** насчитывается **{ps.get('stagnant_pipes_under_0_3_m_s_count', 0)}** участков, где замедленный водообмен может ухудшать качество воды:"
    )
    for s_item in ps.get("top_stagnant_pipes", [])[:4]:
        ans_3.append(
            f"  - Труба `{s_item['pipe_id']}` (диаметр **{s_item['diameter_mm']} мм**, длина **{s_item['length_m']} м**): "
            f"макс. скорость **{s_item['velocity_max_m_s']} м/с**, время пробега воды **{s_item['travel_time_min']} мин**."
        )
    ans_3.append(
        f"- **Обратные токи (`Q < 0` относительно направления `Начало -> Конец`):** зафиксированы на **{ps.get('reverse_flow_pipes_count', 0)}** участках."
    )
    for r_item in ps.get("top_reverse_flow_pipes", [])[:4]:
        ans_3.append(
            f"  - Труба `{r_item['pipe_id']}`: обратное направление держится **{r_item['reverse_steps_of_1440']}** шагов из `1440` (смен знака за сутки: **{r_item['direction_sign_flips_per_day']}**)."
        )
    add_ex(
        "pipe_velocity_and_stagnation_audit",
        f"Оцени скоростной режим трубопроводов модели «{cname}»: есть ли перегруженные водоводы, застойные зоны и участки с реверсом потока?",
        pipes_payload,
        ans_3,
    )

    # --- 4. Аудит насосного оборудования и энергозатрат ---
    pumps_payload = {"city_name": cname, "pumps_summary": pms}
    ans_4 = [
        f"### Режим работы и энергоаудит насосов («{cname}»)",
        "",
        f"- Всего в схеме числится **{pms.get('total_pumps_declared', 0)}** насосных агрегатов: фактически работали **{pms.get('active_pumps_count', 0)}**, находились в резерве (подача `0` все сутки) — **{pms.get('standby_off_pumps_count', 0)}**.",
        f"- Суммарная суточная подача составила **{pms.get('total_pumped_volume_m3_day', 0.0)} м³/сут**, суммарная полезная гидравлическая энергия — **{pms.get('total_hydraulic_energy_kwh_day', 0.0)} кВт·ч/сут**.",
    ]
    for pm in pms.get("active_pumps_details", []):
        ans_4.append(
            f"  - Насос `{pm['pump_id']}` (кривая `{pm['curve_id']}`): наработка **{pm['run_hours_per_day']} ч/сут**, "
            f"число пусков — **{pm['starts_per_day']}**, средняя подача **{pm['flow_mean_m3_h']} м³/ч** (пиковая **{pm['flow_max_m3_h']} м³/ч**), "
            f"расчётный напор `TDH` = **{pm['tdh_head_m']} м**, средняя гидравлическая мощность **{pm['hydraulic_power_kw']} кВт** (**{pm['hydraulic_energy_kwh_day']} кВт·ч/сут**)."
        )
    add_ex(
        "pump_station_energy_audit",
        f"Проведи аудит работы насосных агрегатов города «{cname}»: сколько насосов в резерве, какова цикличность пусков и гидравлическая мощность?",
        pumps_payload,
        ans_4,
    )

    # --- 5. Индивидуальные карточки по конкретным элементам сети (множит число качественных примеров!) ---
    for d_node in ns.get("worst_deficit_nodes", [])[:4]:
        single_payload = {"city_name": cname, "node_diagnostic": d_node}
        add_ex(
            "single_node_diagnosis",
            f"Почему узел {d_node['node_id']} в модели «{cname}» попал в список проблемных и какие меры нужны?",
            single_payload,
            [
                f"Узел `{d_node['node_id']}` в городе «{cname}» обслуживает застройку этажностью **{d_node['floors']} эт.**, для которой требуется минимальный свободный напор **{d_node['required_pressure_m']} м**.",
                f"Фактический минимальный свободный напор на узле опускается до **{d_node['free_pressure_min_m']} м**, что создаёт дефицит **{d_node['margin_min_m']} м** в течение **{d_node['deficit_minutes_per_day']} мин/сут** (при суточном водоразборе узла **{d_node['daily_demand_m3']} м³**).",
                "Рекомендуется проверить потери напора на подводящих участках в часы пикового водоразбора и настройку напорного режима питающей насосной станции.",
            ],
        )

    for s_pipe in ps.get("top_stagnant_pipes", [])[:4]:
        single_p_payload = {"city_name": cname, "pipe_diagnostic": s_pipe}
        add_ex(
            "single_pipe_diagnosis",
            f"Дай оценку гидравлического режима участка трубы {s_pipe['pipe_id']} в городе «{cname}».",
            single_p_payload,
            [
                f"Участок трубопровода `{s_pipe['pipe_id']}` (диаметр **{s_pipe['diameter_mm']} мм**, длина **{s_pipe['length_m']} м**) работает в малоскоростном режиме: максимальная за сутки скорость составляет всего **{s_pipe['velocity_max_m_s']} м/с** (ниже порога самоочищения и циркуляции `0.3 м/с`).",
                f"Среднее время пробега воды по данному участку достигает **{s_pipe['travel_time_min']} мин**. При завышенном диаметре рекомендуется проверить целесообразность кольцевания, переключения задвижек или регламентной промывки участка.",
            ],
        )

    # --- 6. Защитный контр-вопрос против выдумывания отсутствующих цифр (Anti-Hallucination Refusal) ---
    refusal_payload = {
        "city_name": cname,
        "pumps_summary": {
            "active_pumps_count": pms.get("active_pumps_count", 0),
            "total_hydraulic_energy_kwh_day": pms.get("total_hydraulic_energy_kwh_day", 0.0),
        },
    }
    add_ex(
        "anti_hallucination_guard",
        f"Рассчитай точную стоимость электроэнергии в рублях и фактический электрический КПД насосов в городе «{cname}», а также укажи остаточный хлор на тупиковых узлах.",
        refusal_payload,
        [
            f"По переданным фактам модели «{cname}» известна только полезная **гидравлическая** энергия, переданная воде (**{pms.get('total_hydraulic_energy_kwh_day', 0.0)} кВт·ч/сут** для **{pms.get('active_pumps_count', 0)}** работающих насосов).",
            "Точно определить электрическую мощность из сети, стоимость электроэнергии в рублях и концентрацию остаточного хлора по текущему пакету данных **невозможно**, так как во входной выгрузке отсутствуют:",
            "1. Паспортная кривая КПД насоса и электродвигателя (`Кривая КПД`);",
            "2. Тариф на электроэнергию (руб./кВт·ч);",
            "3. Параметры модели качества воды (дозирование реагента на источнике и коэффициенты распада хлора в воде и на стенках труб).",
            "Я не выдумываю недостающие коэффициенты — добавьте кривую КПД и тариф для расчёта фактического энергопотребления.",
        ],
    )

    return examples


RAW_DATASET_EXAMPLES: list[dict[str, Any]] = []
for cname, cfacts in ALL_CITY_FACTS.items():
    RAW_DATASET_EXAMPLES.extend(build_training_examples_for_city(cfacts))

print(f"✅ Сгенерировано обучающих диалогов по вашим городам: {len(RAW_DATASET_EXAMPLES)}")

# --- Дополнительно формируем расширенный пул неразмеченных задач специально для сильной Модели-Учителя ---
import hashlib

TEACHER_SYSTEM_PROMPT = (
    "Ты — главный инженер-гидравлик, эксперт по анализу расчётов отечественного программного комплекса «Аквабит» (AquaBit) "
    "(с опорой на СП 31.13330.2021 и руководство Walski et al.). "
    "Напиши глубокий, профессиональный инженерный разбор результатов расчёта в ПК «Аквабит».\n\n"
    "ЖЕЛЕЗНЫЕ ПРАВИЛА ДИСТИЛЛЯЦИИ (нарушение бракует пример!):\n"
    "1. Опирайся ИСКЛЮЧИТЕЛЬНО на переданный JSON `facts_inline`. Всегда упоминай программный комплекс «Аквабит» (AquaBit) и его CSV-таблицы (НЕ упоминай *.inp файлы EPANET).\n"
    "2. Учитывай, что решатель ПК «Аквабит» уже посчитал гидравлику на полной связной схеме, а `zero_cleaning_summary` отражает только очистку аналитической выборки от транзитных узлов и резервных насосов.\n"
    "3. КАТЕГОРИЧЕСКИ ЗАПРЕЩЕНО считать в уме (не вычисляй проценты, разности или суммы, которых нет в JSON) "
    "и запрещено писать любые числа, отсутствующие во входном `facts_inline` "
    "(разрешены только константы нормативов: 10 м и +4 м на этаж по СП 31.13330.2021, 60 м макс. напор, 0.3 / 1.5 / 3.0 м/с скорости, 24 ч, 1440 шагов).\n"
    "4. Обязательно указывай конкретные строковые ID узлов (c*), труб (e*), насосов (p*), кривых (u*, t*) и объясняй физику процессов.\n"
    "5. Если спрашивают данные, которых нет в `facts_inline` (КПД двигателя, тариф в рублях, остаточный хлор), "
    "чётко перечисли недостающие таблицы в выгрузке ПК «Аквабит» и НЕ выдумывай оценочных цифр."
)


def generate_teacher_prompts_for_city(facts: dict[str, Any]) -> list[dict[str, Any]]:
    cname = facts["city_name"]
    ns = facts.get("nodes_summary", {})
    ps = facts.get("pipes_summary", {})
    pms = facts.get("pumps_summary", {})
    zs = facts.get("zero_cleaning_summary", {})
    qflags = facts.get("data_quality_flags", [])

    compact_overview = {
        "city_name": cname,
        "zero_cleaning_summary": zs,
        "nodes_overview": {k: v for k, v in ns.items() if not isinstance(v, list)},
        "pipes_overview": {k: v for k, v in ps.items() if not isinstance(v, list)},
        "pumps_overview": {k: v for k, v in pms.items() if not isinstance(v, list)},
        "data_quality_flags": qflags,
    }

    specs: list[tuple[str, str, dict[str, Any]]] = [
        ("teacher_exec_memo", f"Подготовь аналитическую записку главному инженеру водоканала по расчёту сети «{cname}»: оцени баланс, главные гидравлические риски и очерёдность мероприятий.", compact_overview),
        ("teacher_tu_developer", f"Застройщик просит техусловия на подключение нового жилого дома в городе «{cname}». Судя по свободным напорам и этажности существующих узлов, какие ограничения и риски просадки давления есть в сети?", {"city_name": cname, "nodes_summary": ns}),
        ("teacher_prv_zoning", f"Проанализируй зоны минимального и избыточного (>60 м) свободного напора в «{cname}». Где целесообразно зонирование сети или установка регуляторов давления PRV?", {"city_name": cname, "nodes_summary": ns}),
        ("teacher_water_hammer", f"Оцени трубопроводы города «{cname}» по критериям книги Walski: какие участки работают с повышенной скоростью, риском гидравлического удара (>3.0 м/с) и частыми реверсами потока?", {"city_name": cname, "pipes_summary": ps}),
        ("teacher_stagnation_flushing", f"Выдели застойные участки сети «{cname}» со скоростью ниже 0.3 м/с и большим временем пробега воды. Как это влияет на качество воды и где нужна промывка или переключение задвижек?", {"city_name": cname, "pipes_summary": ps}),
        ("teacher_pump_vfd_audit", f"Проведи энергоаудит насосов модели «{cname}»: сколько агрегатов в резерве, какова наработка, число пусков и полезная гидравлическая мощность? Оцени целесообразность ЧРП.", {"city_name": cname, "pumps_summary": pms, "excess_nodes_over_60m": ns.get("excess_pressure_over_60m_nodes_count", 0)}),
        ("teacher_dispatcher_shift", f"Объясни сменному диспетчеру особенности суточного режима сети «{cname}»: где меняет направление поток в трубах, как включаются насосы и какие узлы требуют контроля?", {"city_name": cname, "top_reverse_flow_pipes": ps.get("top_reverse_flow_pipes", []), "pumps_summary": pms, "worst_deficit_nodes": ns.get("worst_deficit_nodes", [])}),
        ("teacher_schema_quality", f"Проверь качество расчётной схемы «{cname}» после удаления нулевых строк: сколько транзитных узлов и резервных насосов отфильтровано и есть ли ошибки топологии Начало==Конец?", {"city_name": cname, "zero_cleaning_summary": zs, "schedules_summary": facts.get("schedules_summary", {}), "self_loop_topology_pipes_count": ps.get("self_loop_topology_pipes_count", 0), "data_quality_flags": qflags}),
        ("teacher_refusal_trap", f"Рассчитай для водоканала «{cname}» годовую экономию в рублях при замене насосов, их фактический электрический КПД и остаточный хлор на крайних узлах.", {"city_name": cname, "pumps_summary": {"active_pumps_count": pms.get("active_pumps_count", 0), "total_hydraulic_energy_kwh_day": pms.get("total_hydraulic_energy_kwh_day", 0.0)}}),
    ]

    for dn in ns.get("worst_deficit_nodes", [])[:3]:
        specs.append(("teacher_node_deep_dive", f"Разбери детально причины нехватки свободного напора на узле {dn['node_id']} в городе «{cname}» и предложи инженерные решения.", {"city_name": cname, "node_diagnostic": dn}))
    for sp in ps.get("top_stagnant_pipes", [])[:3]:
        specs.append(("teacher_pipe_deep_dive", f"Проанализируй гидравлический режим участка трубы {sp['pipe_id']} в модели «{cname}» (скорость, диаметр, время пробега воды) и дай рекомендации.", {"city_name": cname, "pipe_diagnostic": sp}))

    out_prompts = []
    for t_type, q_str, p_load in specs:
        pid = f"{cname}_{t_type}_" + hashlib.sha1(f"{cname}::{t_type}::{q_str}".encode("utf-8")).hexdigest()[:10]
        user_msg = f"Вопрос инженера: {q_str}\n\nФакты расчёта (facts_inline):\n```json\n{json.dumps(p_load, ensure_ascii=False, indent=2)}\n```"
        out_prompts.append({
            "prompt_id": pid,
            "city": cname,
            "task_type": t_type,
            "question": q_str,
            "facts_inline": p_load,
            "user_message": user_msg,
        })
    return out_prompts


TEACHER_PROMPTS_POOL: list[dict[str, Any]] = []
for cname, cfacts in ALL_CITY_FACTS.items():
    city_t_prompts = generate_teacher_prompts_for_city(cfacts)
    TEACHER_PROMPTS_POOL.extend(city_t_prompts)

    # Сохраняем удобный Markdown-файл с задачами по городу (если захочется разметить вручную в веб-чате Claude/GPT без API!)
    md_batch_lines = [
        f"# Пакет задач для разметки Моделью-Учителем (город: {cname})",
        "",
        f"**Системная инструкция Учителю:**\n> {TEACHER_SYSTEM_PROMPT}",
        "",
    ]
    for idx_p, tp in enumerate(city_t_prompts, start=1):
        md_batch_lines.extend([f"---", f"## Задача {idx_p} (`{tp['prompt_id']}`)", "", tp["user_message"], ""])
    (WEB_BATCHES_DIR / f"teacher_batch_{cname}.md").write_text("\n".join(md_batch_lines), encoding="utf-8")

unlabeled_pool_path = DATASETS_DIR / "teacher_unlabeled_prompts.jsonl"
with unlabeled_pool_path.open("w", encoding="utf-8") as f:
    for tp in TEACHER_PROMPTS_POOL:
        f.write(json.dumps(tp, ensure_ascii=False) + "\n")

print(f"🎯 Подготовлено неразмеченных глубоких задач для Модели-Учителя: {len(TEACHER_PROMPTS_POOL)} -> {unlabeled_pool_path.name}")
print(f"📝 Карточки для веб-чата (если размечать без API-ключа) сохранены в: {WEB_BATCHES_DIR.resolve()}")

## Клетка 5. Дистилляция знаний от Сильной Модели-Учителя (`Teacher API + Автоисправление цифр`)
Эта клетка связывается с выбранной вами сильной Моделью-Учителем (`Qwen3.5-397B` / `DeepSeek-V3.2` / `Claude Opus` / `GPT-5` / локальный `Ollama`) и размечает `TEACHER_PROMPTS_POOL`:
1. **Защита от потери денег и прогресса:** каждый принятый ответ сразу дописывается в кэш `training_epanet_ai/datasets/teacher_distilled_cache.jsonl` — при повторном запуске уже размеченные задачи берутся из кэша мгновенно!
2. **Цикл самоисправления Учителя (`Critique & Retry`):** каждый ответ Учителя проверяется на числа. Если Учитель случайно посчитал что-то в уме и написал цифру не из `facts_inline`, ноутбук **автоматически отправляет ему список лишних цифр и просит переписать ответ чисто по фактам**!
3. Чтобы запустить разметку, просто впишите `TEACHER_API_KEY` и поставьте `RUN_TEACHER_DISTILLATION = True` в Клетке 1 (или прямо ниже).

In [ ]:
ALLOWED_DOMAIN_CONSTANTS = {0.0, 0.3, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 9.0, 10.0, 14.0, 18.0, 22.0, 24.0, 26.0, 60.0, 1440.0, 2000.0, 4000.0}


def collect_numbers_from_json(obj: Any, out: set[float] | None = None) -> set[float]:
    """Собирает все числовые значения из входного JSON фактов."""
    if out is None:
        out = set()
    if isinstance(obj, bool) or obj is None:
        return out
    if isinstance(obj, (int, float)):
        if not math.isnan(float(obj)):
            out.add(round(float(obj), 6))
        return out
    if isinstance(obj, str):
        for m in re.findall(r"-?\d+(?:\.\d+)?", obj):
            out.add(round(float(m), 6))
        return out
    if isinstance(obj, dict):
        for k, v in obj.items():
            collect_numbers_from_json(k, out)
            collect_numbers_from_json(v, out)
    elif isinstance(obj, (list, tuple)):
        for item in obj:
            collect_numbers_from_json(item, out)
    return out


def verify_no_hallucinated_numbers(assistant_text: str, facts_inline: dict[str, Any]) -> tuple[bool, list[str]]:
    """Проверяет, что каждое число в ответе подтверждено входным facts_inline (без устного счёта и выдумок)."""
    known_nums = collect_numbers_from_json(facts_inline) | ALLOWED_DOMAIN_CONSTANTS
    unverified: list[str] = []
    cleaned_txt = re.sub(r"\b[ceputvR]\d+\b", " ", assistant_text)
    cleaned_txt = re.sub(r"(?m)^\s*\d+[\.\)]\s+", " ", cleaned_txt)
    for token in re.findall(r"(?<![\w.])-?\d+(?:\.\d+)?(?![\w.])", cleaned_txt):
        val = round(float(token), 6)
        if not any(abs(val - kn) <= 1e-3 for kn in known_nums):
            unverified.append(token)
    return (len(unverified) == 0, unverified)


def strip_think_tags(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def check_openrouter_key_status() -> None:
    """Бесплатная проверка статуса ключа OpenRouter (0 токенов, $0.00) перед стартом генерации."""
    import urllib.request

    if "openrouter.ai" not in TEACHER_BASE_URL.lower() or not TEACHER_API_KEY.strip():
        return
    try:
        req = urllib.request.Request(
            "https://openrouter.ai/api/v1/key",
            headers={"Authorization": f"Bearer {TEACHER_API_KEY.strip()}", "User-Agent": "EPANET-AI/1.0"},
        )
        with urllib.request.urlopen(req, timeout=15) as resp:
            kdata = json.loads(resp.read().decode("utf-8")).get("data", {})
        used = float(kdata.get("usage", 0.0) or 0.0)
        rem = kdata.get("limit_remaining")
        rem_str = f"${float(rem):.2f}" if rem is not None else "без лимита ключа"
        print(f"🔑 Ключ OpenRouter проверен | Уже потрачено по ключу: ${used:.4f} | Остаток лимита ключа: {rem_str}")
    except Exception as exc:
        print(f"⚠️ Не удалось проверить статус ключа OpenRouter заранее: {exc}")


def call_teacher_api(messages: list[dict[str, str]]) -> tuple[str, float, int, int]:
    """Вызов OpenRouter / OpenAI-совместимого API с автоотключением лишнего <think> и подсчётом точной стоимости в $."""
    import urllib.error
    import urllib.request

    url = TEACHER_BASE_URL.rstrip("/") + "/chat/completions"
    is_openrouter = "openrouter.ai" in TEACHER_BASE_URL.lower()

    payload_dict: dict[str, Any] = {
        "model": TEACHER_MODEL_ID,
        "messages": messages,
        "temperature": 0.2,
        "max_tokens": 1800,
    }
    if is_openrouter and not TEACHER_ENABLE_REASONING:
        payload_dict["reasoning"] = {"enabled": False}

    headers = {
        "Content-Type": "application/json",
        "User-Agent": "EPANET-AI-Distiller/1.0",
    }
    if TEACHER_API_KEY and TEACHER_API_KEY.strip():
        headers["Authorization"] = f"Bearer {TEACHER_API_KEY.strip()}"
    if is_openrouter:
        headers["HTTP-Referer"] = "https://epanet-ai.local"
        headers["X-Title"] = "EPANET-AI Distillation"

    def _send(body: dict[str, Any]) -> dict[str, Any]:
        req = urllib.request.Request(url, data=json.dumps(body).encode("utf-8"), headers=headers, method="POST")
        with urllib.request.urlopen(req, timeout=150) as resp:
            return json.loads(resp.read().decode("utf-8"))

    try:
        resp_data = _send(payload_dict)
    except urllib.error.HTTPError as http_err:
        # Если выбрана модель с неотключаемым reasoning, повторяем запрос без поля reasoning
        if http_err.code == 400 and "reasoning" in payload_dict:
            fallback_body = dict(payload_dict)
            fallback_body.pop("reasoning", None)
            fallback_body["max_tokens"] = 6144
            resp_data = _send(fallback_body)
        else:
            err_body = http_err.read().decode("utf-8", errors="replace")
            raise RuntimeError(f"HTTP {http_err.code} от API Учителя: {err_body}") from http_err

    raw_txt = resp_data["choices"][0]["message"]["content"] or ""
    usage = resp_data.get("usage", {}) or {}
    in_tok = int(usage.get("prompt_tokens", 0) or 0)
    out_tok = int(usage.get("completion_tokens", 0) or 0)
    if usage.get("cost") is not None:
        call_cost_usd = float(usage["cost"])
    else:
        # Резервная оценка по тарифу Qwen3.5-397B ($0.50/M вход, $3.60/M выход)
        call_cost_usd = (in_tok * 0.50 + out_tok * 3.60) / 1_000_000.0

    return strip_think_tags(raw_txt), call_cost_usd, in_tok, out_tok


TEACHER_CACHE_PATH = DATASETS_DIR / "teacher_distilled_cache.jsonl"
MANUAL_TEACHER_IMPORT_PATH = DATASETS_DIR / "teacher_manual_import.jsonl"

DISTILLED_TEACHER_EXAMPLES: list[dict[str, Any]] = []
cached_teacher_map: dict[str, dict[str, Any]] = {}

# 1. Подгружаем уже размеченные ранее ответы Учителя из кэша и ручного импорта (если есть)
for existing_path in (TEACHER_CACHE_PATH, MANUAL_TEACHER_IMPORT_PATH):
    if existing_path.exists():
        for line in existing_path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line)
                if "prompt_id" in rec:
                    cached_teacher_map[rec["prompt_id"]] = rec

# 2. Если включён RUN_TEACHER_DISTILLATION = True, размечаем новые задачи через API Учителя с контролем бюджета!
if RUN_TEACHER_DISTILLATION:
    is_local_teacher = ("localhost" in TEACHER_BASE_URL) or ("127.0.0.1" in TEACHER_BASE_URL)
    if not is_local_teacher and not TEACHER_API_KEY.strip():
        print("⚠️ Укажите ваш ключ sk-or-v1-... в переменной TEACHER_API_KEY в Клетке 1!")
    else:
        check_openrouter_key_status()
        accepted_1st = 0
        accepted_retry = 0
        rejected_teacher = 0
        from_cache_cnt = 0
        session_cost_usd = 0.0
        session_in_tokens = 0
        session_out_tokens = 0

        for idx_t, task in enumerate(TEACHER_PROMPTS_POOL, start=1):
            pid = task["prompt_id"]
            if pid in cached_teacher_map:
                from_cache_cnt += 1
                continue

            if session_cost_usd >= TEACHER_MAX_BUDGET_USD:
                print(
                    f"🛑 Достигнут защитный порог бюджета TEACHER_MAX_BUDGET_USD = ${TEACHER_MAX_BUDGET_USD:.2f}. "
                    "Все уже размеченные ответы надёжно сохранены в teacher_distilled_cache.jsonl!"
                )
                break

            dialog = [
                {"role": "system", "content": TEACHER_SYSTEM_PROMPT},
                {"role": "user", "content": task["user_message"]},
            ]
            valid_ans = None
            retried = False
            bad_tokens: list[str] = []
            task_cost = 0.0

            for attempt in range(MAX_TEACHER_RETRIES_ON_HALLUCINATION + 1):
                ans_candidate, c_usd, in_t, out_t = call_teacher_api(dialog)
                session_cost_usd += c_usd
                task_cost += c_usd
                session_in_tokens += in_t
                session_out_tokens += out_t

                ok_num, bad_tokens = verify_no_hallucinated_numbers(ans_candidate, task["facts_inline"])
                if ok_num and len(ans_candidate) >= 40:
                    valid_ans = ans_candidate
                    retried = attempt > 0
                    break
                dialog.append({"role": "assistant", "content": ans_candidate})
                dialog.append({
                    "role": "user",
                    "content": (
                        f"В твоём ответе найдены числа {bad_tokens}, отсутствующие во входном JSON `facts_inline`. "
                        "Устный счёт и округление запрещены! Перепиши ответ целиком, оставив ТОЛЬКО точные числа из `facts_inline`."
                    ),
                })

            if valid_ans is not None:
                if retried:
                    accepted_retry += 1
                else:
                    accepted_1st += 1
                distilled_rec = {
                    "prompt_id": pid,
                    "city": task["city"],
                    "task_type": task["task_type"],
                    "source": f"teacher:{TEACHER_MODEL_ID}",
                    "facts_inline": task["facts_inline"],
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT_EPANET_AI},
                        {"role": "user", "content": task["user_message"]},
                        {"role": "assistant", "content": valid_ans},
                    ],
                }
                cached_teacher_map[pid] = distilled_rec
                with TEACHER_CACHE_PATH.open("a", encoding="utf-8") as cf:
                    cf.write(json.dumps(distilled_rec, ensure_ascii=False) + "\n")
                status_lbl = "после автоисправления" if retried else "с 1-й попытки"
                print(
                    f"  [{idx_t}/{len(TEACHER_PROMPTS_POOL)}] ✅ {pid} ({status_lbl}) "
                    f"| +${task_cost:.4f} | Всего: ${session_cost_usd:.4f}"
                )
            else:
                rejected_teacher += 1
                print(
                    f"  [{idx_t}/{len(TEACHER_PROMPTS_POOL)}] ❌ {pid} отклонён (неподтверждённые числа: {bad_tokens}) "
                    f"| Всего: ${session_cost_usd:.4f}"
                )

        print(
            f"🎓 Итог сессии Учителя: с 1-й попытки={accepted_1st}, после самоисправления={accepted_retry}, "
            f"из кэша={from_cache_cnt}, отклонено={rejected_teacher} | "
            f"Токены: in={session_in_tokens:,}, out={session_out_tokens:,} | Итого расход: ${session_cost_usd:.4f}"
        )
else:
    print(
        f"ℹ️ Онлайн-вызов API Учителя сейчас выключен (ключ не указан). "
        f"В кэше Учителя уже лежит ответов: {len(cached_teacher_map)}."
    )
    print("Чтобы запустить разметку — вставьте ваш sk-or-v1-... ключ в Клетке 1 и нажмите Run All!")

DISTILLED_TEACHER_EXAMPLES = list(cached_teacher_map.values())


## Клетка 6. Слияние дистиллированных ответов Учителя и эталонных примеров в финальные `train.jsonl` / `val.jsonl`
Собирает итоговый обучающий корпус для вашей локальной модели-Ученика (`Qwen/Qwen3.5-4B`):
- берёт все прошедшие проверку ответы **Модели-Учителя** (`DISTILLED_TEACHER_EXAMPLES`) и (при `INCLUDE_SEED_EXAMPLES_WITH_TEACHER = True` или если кэш Учителя пока пуст) дополняет их базовыми проверочными примерами;
- повторно проверяет 100% примеров анти-галлюцинационным фильтром и сохраняет **`train.jsonl`**, **`val.jsonl`** и **`train_alpaca.json`**.

In [ ]:
CANDIDATE_CORPUS: list[dict[str, Any]] = []
CANDIDATE_CORPUS.extend(DISTILLED_TEACHER_EXAMPLES)
if INCLUDE_SEED_EXAMPLES_WITH_TEACHER or not DISTILLED_TEACHER_EXAMPLES:
    for ex in RAW_DATASET_EXAMPLES:
        ex_copy = dict(ex)
        ex_copy.setdefault("source", "deterministic_seed")
        CANDIDATE_CORPUS.append(ex_copy)

VERIFIED_EXAMPLES: list[dict[str, Any]] = []
rejected_count = 0

for ex in CANDIDATE_CORPUS:
    ans_txt = ex["messages"][-1]["content"]
    ok, bad_nums = verify_no_hallucinated_numbers(ans_txt, ex["facts_inline"])
    if ok:
        VERIFIED_EXAMPLES.append(ex)
    else:
        rejected_count += 1
        print(f"⚠️ Отклонён пример ({ex['city']} / {ex['task_type']}), неподтверждённые числа: {bad_nums}")

rng = random.Random(RANDOM_SEED)
shuffled = list(VERIFIED_EXAMPLES)
rng.shuffle(shuffled)

n_val = max(1, int(round(len(shuffled) * VAL_SPLIT_RATIO))) if len(shuffled) >= 4 else 0
val_items = shuffled[:n_val]
train_items = shuffled[n_val:]

train_jsonl_path = DATASETS_DIR / "train.jsonl"
val_jsonl_path = DATASETS_DIR / "val.jsonl"
alpaca_json_path = DATASETS_DIR / "train_alpaca.json"

with train_jsonl_path.open("w", encoding="utf-8") as f:
    for item in train_items:
        f.write(json.dumps({
            "messages": item["messages"],
            "city": item["city"],
            "task_type": item["task_type"],
            "source": item.get("source", "seed"),
        }, ensure_ascii=False) + "\n")

with val_jsonl_path.open("w", encoding="utf-8") as f:
    for item in val_items:
        f.write(json.dumps({
            "messages": item["messages"],
            "city": item["city"],
            "task_type": item["task_type"],
            "source": item.get("source", "seed"),
        }, ensure_ascii=False) + "\n")

alpaca_records = [
    {
        "instruction": item["messages"][0]["content"],
        "input": item["messages"][1]["content"],
        "output": item["messages"][2]["content"],
    }
    for item in train_items
]
alpaca_json_path.write_text(json.dumps(alpaca_records, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

teacher_share = sum(1 for x in VERIFIED_EXAMPLES if str(x.get("source", "")).startswith("teacher"))
print(f"✅ Итоговый проверенный датасет: {len(VERIFIED_EXAMPLES)} диалогов (из них от Модели-Учителя: {teacher_share}, базовых: {len(VERIFIED_EXAMPLES) - teacher_share}, отклонено: {rejected_count})")
print(f"📁 Обучающая выборка Ученика (train.jsonl) : {len(train_items)} -> {train_jsonl_path.resolve()}")
print(f"📁 Проверочная выборка Ученика (val.jsonl) : {len(val_items)} -> {val_jsonl_path.resolve()}")

## Клетка 7. Обучение быстрого Табличного ИИ-ранжировщика рисков (`Tabular ML Risk Classifier`)
Этот модуль обучается **мгновенно прямо сейчас даже на обычном ноутбуке без видеокарты**:
- собирает все очищенные узлы и участки труб всех ваших городов из `data_cleaned/`;
- формирует матрицу инженерных признаков (`Этажность_дома`, `Свободный_напор_мин_м`, `Запас_напора_мин_м`, `Скорость_макс_м_с`, `Время_пробега_воды_мин`, `Шагов_обратного_потока`);
- обучает модель предсказания и ранжирования индекса риска (`RandomForest` / `GradientBoosting` из `scikit-learn`, либо встроенный векторизованный логистический классификатор на чистом `NumPy`), сохраняет обученные веса в `training_epanet_ai/models/` и выводит рейтинг самых уязвимых элементов сети!

In [ ]:
def train_tabular_pipe_and_node_risk_ai(cleaned_cities: list[Path], models_out: Path) -> dict[str, Any]:
    """Обучает табличную модель оценки риска по всем трубам и узлам корпуса городов."""
    pipe_frames = []
    node_frames = []

    for cdir in cleaned_cities:
        cname = unicodedata.normalize("NFC", cdir.name)
        ddir = resolve_subdir_unicode(cdir, "расчетные_параметры")
        fp = ddir / "Труба_скорости_и_режимы_Walski.csv"
        fn = ddir / "Узел_этажность_и_свободный_напор.csv"
        dfp = safe_read_csv(fp)
        if not dfp.empty:
            dfp["Город"] = cname
            pipe_frames.append(dfp)
        dfn = safe_read_csv(fn)
        if not dfn.empty:
            dfn["Город"] = cname
            node_frames.append(dfn)

    result_info: dict[str, Any] = {}
    if not pipe_frames:
        return result_info

    all_pipes = pd.concat(pipe_frames, ignore_index=True)
    feat_cols = ["Диаметр_мм", "Длина_м", "Скорость_макс_м_с", "Время_пробега_воды_мин", "Шагов_обратного_потока"]
    for col in feat_cols:
        if col not in all_pipes.columns:
            all_pipes[col] = 0.0

    X_raw = all_pipes[feat_cols].fillna(0.0).to_numpy(dtype=np.float64)
    # Целевая метка эксплуатационного риска участка (перегрузка >1.8 м/с, сильный застой <0.20 м/с при длине >80 м, или частый реверс)
    v_arr = all_pipes["Скорость_макс_м_с"].fillna(0.0).to_numpy()
    l_arr = all_pipes["Длина_м"].fillna(0.0).to_numpy()
    rev_arr = all_pipes["Шагов_обратного_потока"].fillna(0.0).to_numpy()
    y_risk = ((v_arr > 1.8) | ((v_arr < 0.20) & (l_arr > 50.0)) | (rev_arr > 360)).astype(int)

    # Нормализация признаков (Z-score) для прозрачности и переносимости
    means = X_raw.mean(axis=0)
    stds = np.where(X_raw.std(axis=0) > 1e-9, X_raw.std(axis=0), 1.0)
    X_norm = (X_raw - means) / stds

    if HAS_SKLEARN and len(np.unique(y_risk)) >= 2 and len(all_pipes) >= 6:
        import pickle
        from sklearn.ensemble import RandomForestClassifier
        clf = RandomForestClassifier(n_estimators=120, max_depth=6, random_state=RANDOM_SEED)
        clf.fit(X_norm, y_risk)
        risk_probs = clf.predict_proba(X_norm)[:, 1]
        importances = dict(zip(feat_cols, [round(float(x), 4) for x in clf.feature_importances_]))
        with (models_out / "tabular_pipe_risk_rf.pkl").open("wb") as f:
            pickle.dump({"model": clf, "feature_columns": feat_cols, "means": means.tolist(), "stds": stds.tolist()}, f)
        engine_used = "scikit-learn RandomForestClassifier (120 деревьев)"
    else:
        # Чистый градиентный спуск логистической регрессии с L2-регуляризацией на NumPy (работает без внешних библиотек!)
        w = np.zeros(X_norm.shape[1], dtype=np.float64)
        b = 0.0
        target = y_risk.astype(np.float64) if len(np.unique(y_risk)) >= 2 else np.clip(v_arr / 2.5 + rev_arr / 1440.0, 0.0, 1.0)
        for _ in range(300):
            logits = np.clip(X_norm @ w + b, -30.0, 30.0)
            preds = 1.0 / (1.0 + np.exp(-logits))
            grad_w = (X_norm.T @ (preds - target)) / len(X_norm) + 0.01 * w
            grad_b = float(np.mean(preds - target))
            w -= 0.15 * grad_w
            b -= 0.15 * grad_b
        risk_probs = 1.0 / (1.0 + np.exp(-np.clip(X_norm @ w + b, -30.0, 30.0)))
        abs_w = np.abs(w)
        norm_w = abs_w / max(1e-9, float(np.sum(abs_w)))
        importances = dict(zip(feat_cols, [round(float(x), 4) for x in norm_w]))
        engine_used = "Встроенная регуляризованная логистическая модель (NumPy)"

    all_pipes["Индекс_риска_ИИ"] = np.round(risk_probs, 4)
    ranked_pipes = all_pipes.sort_values("Индекс_риска_ИИ", ascending=False)
    ranked_out = models_out / "ranked_pipes_by_ai_risk.csv"
    ranked_pipes.to_csv(ranked_out, index=False, encoding="utf-8-sig")

    meta_path = models_out / "tabular_risk_model_meta.json"
    meta_payload = {
        "engine": engine_used,
        "pipes_trained_on": int(len(all_pipes)),
        "feature_importances": importances,
        "ranked_output_csv": str(ranked_out.resolve()),
    }
    meta_path.write_text(json.dumps(meta_payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    result_info["engine"] = engine_used
    result_info["pipes_count"] = len(all_pipes)
    result_info["importances"] = importances
    result_info["top_pipes_df"] = ranked_pipes[["Город", "ID", "Диаметр_мм", "Длина_м", "Скорость_макс_м_с", "Шагов_обратного_потока", "Индекс_риска_ИИ"]].head(10)
    return result_info


TABULAR_AI_RESULT = train_tabular_pipe_and_node_risk_ai(CLEANED_CITY_DIRS, MODELS_DIR)
if TABULAR_AI_RESULT:
    print(f"✅ Табличный ИИ-ранжировщик обучен на {TABULAR_AI_RESULT['pipes_count']} участках труб ({TABULAR_AI_RESULT['engine']})")
    print(f"Важность признаков: {TABULAR_AI_RESULT['importances']}")
    display(TABULAR_AI_RESULT["top_pipes_df"])

## Клетка 8. (Опционально) Автоустановка библиотек для локального QLoRA-обучения LLM без консоли
Если вы хотите запустить нейросетевое дообучение `Qwen3` прямо на вашем компьютере, а библиотеки `transformers`, `peft`, `trl`, `datasets`, `accelerate` ещё не установлены — просто переключите `INSTALL_LLM_LIBRARIES = True` ниже и запустите клетку (`Shift + Enter`). Установка пройдёт прямо внутри текущего ядра Python без открытия PowerShell.

In [ ]:
if INSTALL_LLM_LIBRARIES:
    # 1. ВАЖНО для Windows: если есть карта NVIDIA, СНАЧАЛА ставим CUDA-сборку torch,
    # иначе обычный pip скачает CPU-версию torch+cpu и отключит видеокарту!
    if HAS_NVIDIA_HW and not HAS_CUDA:
        idx_url = f"https://download.pytorch.org/whl/{DETECTED_CUDA_WHEEL_TAG}"
        print(f"⏳ Шаг 1/2: Устанавливаю CUDA-версию PyTorch ({DETECTED_CUDA_WHEEL_TAG}) для {PRIMARY_NVIDIA_GPU['name']}...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torch", "--index-url", idx_url,
        ])

    pkgs = ["transformers>=4.57.0", "peft>=0.15.0", "trl>=0.16.0", "datasets>=3.2.0", "accelerate>=1.4.0"]
    if platform.system() in {"Linux", "Windows"} and HAS_NVIDIA_HW:
        pkgs.append("bitsandbytes>=0.45.0")
    print("⏳ Шаг 2/2: Устанавливаю библиотеки Transformers + PEFT + TRL для QLoRA-обучения...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
    print("✅ Установка завершена! Нажмите 'Restart' (Перезапустить ядро) и выполните ноутбук, чтобы активировать GPU.")
else:
    print("ℹ️ Автоустановка пропущена (INSTALL_LLM_LIBRARIES = False). Поставьте True, если нужно доустановить пакеты.")


## Клетка 9. Дообучение языковой модели `Qwen3.5-4B` на вашем `train.jsonl` (QLoRA / PEFT)
Когда будете готовы запустить само нейросетевое обучение (на вашей видеокарте NVIDIA или в Google Colab), переключите **`RUN_LLM_TRAINING_NOW = True`** в Клетке 1 (или прямо ниже) и запустите эту клетку:
- при наличии `unsloth` + CUDA включается ускоренное 4-битное QLoRA-обучение;
- на стандартном Windows-стеке с GPU включается `HuggingFace Transformers + BitsAndBytes 4-bit + PEFT LoRA + SFTTrainer`;
- обученный адаптер сохраняется в **`training_epanet_ai/models/epanet_ai_qwen_lora/`**.

In [ ]:
# Раскомментируйте следующую строку, чтобы включить обучение прямо сейчас:
# RUN_LLM_TRAINING_NOW = True

# Защита от зависания загрузки весов Hugging Face на Windows (отключает глючный протокол Xet и чинит текстовый прогресс-бар без ipywidgets):
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
# Раскомментируйте строку ниже, если провайдер режет скорость к huggingface.co:
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com" 

LORA_OUTPUT_DIR = MODELS_DIR / "epanet_ai_qwen_lora"

def run_qwen_qlora_sft(
    model_id: str,
    train_path: Path,
    val_path: Path,
    output_dir: Path,
    *,
    num_epochs: int = 3,
    learning_rate: float = 2e-4,
    max_seq_length: int = 2048,
) -> dict[str, Any]:
    """Запускает дообучение Qwen3 методом QLoRA / LoRA на сгенерированном датасете EPANET-AI."""
    import torch
    from datasets import load_dataset
    from trl import SFTConfig, SFTTrainer

    data_files = {"train": str(train_path)}
    if val_path.exists() and val_path.stat().st_size > 0:
        data_files["validation"] = str(val_path)
    ds = load_dataset("json", data_files=data_files)

    use_cuda = torch.cuda.is_available()
    if check_pkg("unsloth") and use_cuda:
        from unsloth import FastLanguageModel
        print(f"🚀 Запускаем ускоренное 4-bit QLoRA обучение через Unsloth для {model_id}...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_id,
            max_seq_length=max_seq_length,
            load_in_4bit=True,
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=16,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_alpha=32,
            lora_dropout=0.0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=RANDOM_SEED,
        )
    else:
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        print(f"🚀 Запускаем стандартное PEFT LoRA обучение ({'CUDA GPU' if use_cuda else 'CPU'}) для {model_id}...")
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        # Принудительно включаем консольный tqdm-прогресс в мегабайтах (обходит отсутствие ipywidgets в VS Code)
        import tqdm.std
        import huggingface_hub.utils.tqdm as hf_tqdm
        hf_tqdm.tqdm = tqdm.std.tqdm

        model_kwargs: dict[str, Any] = {
            "device_map": {"": 0} if use_cuda else None,
            "low_cpu_mem_usage": True,
        }
        if use_cuda and check_pkg("bitsandbytes"):
            model_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
            )
        model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
        if use_cuda and check_pkg("bitsandbytes"):
            model = prepare_model_for_kbit_training(model)

        if globals().get("CONTINUE_FROM_SAVED_LORA", True) and (output_dir / "adapter_config.json").exists():
            from peft import PeftModel
            print(f"♻️ Найден уже обученный адаптер в {output_dir.name} — продолжаем дообучение поверх него (не с нуля)!")
            model = PeftModel.from_pretrained(model, str(output_dir), is_trainable=True)
        else:
            lora_cfg = LoraConfig(
                r=16,
                lora_alpha=32,
                lora_dropout=0.05,
                bias="none",
                task_type="CAUSAL_LM",
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            )
            model = get_peft_model(model, lora_cfg)

    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    def _render_template(msgs: list[dict[str, str]], add_gen: bool = False) -> str:
        try:
            return tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=add_gen,
                enable_thinking=ENABLE_THINKING_MODE,
            )
        except TypeError:
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=add_gen)

    def format_chat(batch: dict[str, list[Any]]) -> dict[str, list[str]]:
        return {"text": [_render_template(msgs, add_gen=False) for msgs in batch["messages"]]}

    ds_formatted = ds.map(format_chat, batched=True)

    # Автоматическая оптимизация под ноутбучные видеокарты 6–8 ГБ VRAM (например RTX 3060 Laptop 6 ГБ):
    is_compact_vram = bool(use_cuda and VRAM_GB <= 8.5)
    eff_seq_len = min(max_seq_length, 1536) if is_compact_vram else max_seq_length
    eff_batch = 1 if (not use_cuda or is_compact_vram) else 2
    eff_grad_accum = 8 if is_compact_vram else 4
    eff_optim = "paged_adamw_8bit" if (use_cuda and check_pkg("bitsandbytes")) else "adamw_torch"
    if is_compact_vram:
        print(f"⚙️ Включён бережный профиль видеопамяти для {VRAM_GB} ГБ VRAM (batch=1, grad_accum=8, gradient_checkpointing=True, paged_adamw_8bit, seq={eff_seq_len})")

    sft_args = SFTConfig(
        output_dir=str(output_dir),
        num_train_epochs=num_epochs,
        per_device_train_batch_size=eff_batch,
        gradient_accumulation_steps=eff_grad_accum,
        gradient_checkpointing=bool(use_cuda),
        gradient_checkpointing_kwargs={"use_reentrant": False} if use_cuda else None,
        optim=eff_optim,
        learning_rate=learning_rate,
        logging_steps=5,
        save_strategy="epoch",
        eval_strategy="epoch" if "validation" in ds_formatted else "no",
        bf16=bool(use_cuda and torch.cuda.is_bf16_supported()),
        fp16=bool(use_cuda and not torch.cuda.is_bf16_supported()),
        dataset_text_field="text",
        max_length=eff_seq_len,
        report_to="none",
        seed=RANDOM_SEED,
    )

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=ds_formatted["train"],
        eval_dataset=ds_formatted.get("validation"),
        args=sft_args,
    )
    train_res = trainer.train()
    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    metrics = dict(train_res.metrics)
    (output_dir / "training_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")

    # Освобождаем видеопамять RTX 3060 сразу после сохранения адаптера, чтобы Клетка 11 стартовала со свободной VRAM!
    import gc
    del trainer
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics


if RUN_LLM_TRAINING_NOW:
    if not train_jsonl_path.exists() or train_jsonl_path.stat().st_size == 0:
        print("⚠️ Файл train.jsonl пуст — сначала выполните Клетки 3–5.")
    else:
        metrics_out = run_qwen_qlora_sft(
            SELECTED_MODEL_ID,
            train_jsonl_path,
            val_jsonl_path,
            LORA_OUTPUT_DIR,
            num_epochs=int(globals().get("NUM_TRAIN_EPOCHS", 3)),
        )
        print(f"🎉 Дообучение завершено! Адаптер сохранён в: {LORA_OUTPUT_DIR.resolve()}")
        print(metrics_out)
else:
    print(f"ℹ️ Датасет готов к QLoRA-обучению ({len(train_items)} train / {len(val_items)} val диалогов).")
    print("Чтобы запустить нейросетевое обучение, поставьте RUN_LLM_TRAINING_NOW = True и выполните эту клетку.")

## Клетка 10. Создание лёгкого архива `colab_training_bundle.zip` (если на ПК нет мощной видеокарты)
Упаковывает ваши готовые компактные файлы `train.jsonl`, `val.jsonl` и `facts_*.json` (размером в считанные килобайты, **без гигабайтных сырых CSV!**) в один архив `training_epanet_ai/colab_training_bundle.zip`. Его можно за 5 секунд загрузить в бесплатный Google Colab с видеокартой T4 (16 ГБ) и обучить `Qwen3.5-4B` за 10–15 минут!

In [ ]:
COLAB_ZIP_PATH = TRAINING_DIR / "colab_training_bundle.zip"

with zipfile.ZipFile(COLAB_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in (FACTS_DIR, DATASETS_DIR):
        for fpath in sorted(folder.glob("*")):
            if fpath.is_file():
                zf.write(fpath, arcname=f"training_epanet_ai/{folder.name}/{fpath.name}")

print(f"📦 Готов лёгкий архив для обучения на любом облачном или внешнем GPU: {COLAB_ZIP_PATH.resolve()} ({COLAB_ZIP_PATH.stat().st_size / 1024.0:.1f} КБ)")

## Клетка 11. Интерактивная проверка работы ИИ на ваших городах + детектор галлюцинаций
Задайте любой вопрос по любому городу из `ALL_CITY_FACTS`:
- если в `training_epanet_ai/models/epanet_ai_qwen_lora/` уже лежат веса вашей обученной модели — отвечает именно ваша дообученная нейросеть `Qwen3`;
- если веса ещё не обучены — отвечает встроенный эталонный интерпретатор по `facts_<город>.json`;
- после ответа автоматически запускается **`verify_no_hallucinated_numbers()`** и проверяет каждую цифру ответа!

In [ ]:
# Делаем Клетку 11 полностью автономной (чтобы можно было запускать её сразу после Клеток 1–3, пропуская Клетки 4–10!):
if "LORA_OUTPUT_DIR" not in globals():
    LORA_OUTPUT_DIR = MODELS_DIR / "epanet_ai_qwen_lora"

if "SYSTEM_PROMPT_EPANET_AI" not in globals() or "Аквабит" not in SYSTEM_PROMPT_EPANET_AI:
    SYSTEM_PROMPT_EPANET_AI = (
        "Ты — экспертная инженерная ИИ-система интерпретации гидравлических расчётов программного комплекса «Аквабит» (AquaBit) "
        "для водопроводных сетей (с опорой на СП 31.13330.2021 и справочник Walski et al.). "
        "Всегда упоминай в ответе расчётный комплекс «Аквабит» (AquaBit) и его таблицы (Узел.csv, Труба.csv, Насос.csv, Кривая QH.csv, Потребители Узел.csv) — "
        "НЕ называй модель файлом *.inp и НЕ подменяй название ПК «Аквабит» на классический EPANET. "
        "Помни: гидравлический расчёт уже выполнен решателем ПК «Аквабит» на полной связной модели сети, а показатели pruned_* в zero_cleaning_summary "
        "означают лишь фильтрацию транзитных узлов (где Требуемое давление = 0) и выключенных резервных насосов при подготовке аналитической сводки (связность расчёта Аквабит сохранена). "
        "Отвечай строго по числам из переданного JSON (facts_inline): запрещено выдумывать или пересчитывать в уме цифры, отсутствующие во входном JSON."
    )

if "strip_think_tags" not in globals():
    def strip_think_tags(text: str) -> str:
        return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

if "verify_no_hallucinated_numbers" not in globals():
    _ALLOWED_CONSTS = {0.0, 0.3, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 9.0, 10.0, 14.0, 18.0, 22.0, 24.0, 26.0, 60.0, 1440.0, 2000.0, 4000.0}

    def _collect_nums(obj: Any, out: set[float] | None = None) -> set[float]:
        if out is None:
            out = set()
        if isinstance(obj, bool) or obj is None:
            return out
        if isinstance(obj, (int, float)):
            if not math.isnan(float(obj)):
                out.add(round(float(obj), 6))
            return out
        if isinstance(obj, str):
            for m in re.findall(r"-?\d+(?:\.\d+)?", obj):
                out.add(round(float(m), 6))
            return out
        if isinstance(obj, dict):
            for k, v in obj.items():
                _collect_nums(k, out)
                _collect_nums(v, out)
        elif isinstance(obj, (list, tuple)):
            for it in obj:
                _collect_nums(it, out)
        return out

    def verify_no_hallucinated_numbers(assistant_text: str, facts_inline: dict[str, Any]) -> tuple[bool, list[str]]:
        known = _collect_nums(facts_inline) | _ALLOWED_CONSTS
        bad: list[str] = []
        cleaned = re.sub(r"\b[ceputvR]\d+\b", " ", assistant_text)
        cleaned = re.sub(r"(?m)^\s*\d+[\.\)]\s+", " ", cleaned)
        for tok in re.findall(r"(?<![\w.])-?\d+(?:\.\d+)?(?![\w.])", cleaned):
            val = round(float(tok), 6)
            if not any(abs(val - kn) <= 1e-3 for kn in known):
                bad.append(tok)
        return (len(bad) == 0, bad)

# Просто укажите название нужного города строкой (БЕЗ next(iter(...)), который всегда брал 1-й город по алфавиту — «Алматы»!):
PREFERRED_CITY = "Орша"   # Варианты: "Орша", "тенино", "хорезмская", "куляб", "модель семей", "Алматы"
CITY_TO_ASK = PREFERRED_CITY if PREFERRED_CITY in ALL_CITY_FACTS else next(iter(ALL_CITY_FACTS.keys()), "")
if PREFERRED_CITY and PREFERRED_CITY not in ALL_CITY_FACTS and CITY_TO_ASK:
    print(f"ℹ️ Город «{PREFERRED_CITY}» не найден в ALL_CITY_FACTS {list(ALL_CITY_FACTS.keys())}, выбран «{CITY_TO_ASK}».")

USER_QUESTION = "Дай комплексную оценку результатов расчёта в ПК «Аквабит»: оцени напоры по этажности, скорости в трубах, работу насосов и выдели главные риски."

# Глобальный кэш загруженной 4-битной модели в видеопамяти RTX 3060 (чтобы повторные вопросы отвечались за 5–15 секунд без перезагрузки весов!)
if "_EPANET_LORA_ENGINE" not in globals():
    _EPANET_LORA_ENGINE: dict[str, Any] = {}


def _load_trained_qwen_lora_once(adapter_dir: Path) -> tuple[Any, Any]:
    """Загружает базовую Qwen3.5-4B строго в 4-битном режиме NF4 целиком на GPU (без offload на диск и без ошибки language_model KeyError)."""
    import gc
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    cache_key = str(adapter_dir.resolve())
    if _EPANET_LORA_ENGINE.get("key") == cache_key:
        return _EPANET_LORA_ENGINE["model"], _EPANET_LORA_ENGINE["tokenizer"]

    # Освобождаем видеопамять от объектов обучения Клетки 9 (если ядро не перезапускалось после тренировки)
    _EPANET_LORA_ENGINE.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    adapter_cfg = json.loads((adapter_dir / "adapter_config.json").read_text(encoding="utf-8"))
    base_model_id = adapter_cfg.get("base_model_name_or_path") or SELECTED_MODEL_ID

    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

    use_cuda = torch.cuda.is_available()
    print(f"⏳ Загружаю базовую {base_model_id} (4-bit NF4) + ваш обученный адаптер {adapter_dir.name} на {'GPU' if use_cuda else 'CPU'}...")

    tokenizer = AutoTokenizer.from_pretrained(str(adapter_dir))
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs: dict[str, Any] = {"low_cpu_mem_usage": True}
    if use_cuda and check_pkg("bitsandbytes"):
        compute_dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=compute_dt,
        )
        # КРИТИЧЕСКИ ВАЖНО для Qwen3.5 на 6 ГБ VRAM: кладём все 32 слоя (2.7 ГБ в 4-bit) строго на cuda:0!
        # Это полностью исключает offload слоёв 22..31 на диск/meta и ошибку KeyError: language_model в PEFT.
        model_kwargs["device_map"] = {"": 0}
    elif use_cuda:
        model_kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        model_kwargs["device_map"] = {"": 0}

    base_model = AutoModelForCausalLM.from_pretrained(base_model_id, **model_kwargs)
    peft_model = PeftModel.from_pretrained(base_model, str(adapter_dir), is_trainable=False)
    peft_model.eval()

    _EPANET_LORA_ENGINE["key"] = cache_key
    _EPANET_LORA_ENGINE["model"] = peft_model
    _EPANET_LORA_ENGINE["tokenizer"] = tokenizer
    if use_cuda:
        used_gb = round(torch.cuda.memory_allocated(0) / (1024**3), 2)
        print(f"✅ Модель готова! Занято видеопамяти: {used_gb} ГБ VRAM (повторные вопросы будут выполняться мгновенно без перезагрузки).")
    return peft_model, tokenizer


def generate_epanet_ai_answer(city_name: str, question: str) -> tuple[str, str]:
    """Генерирует ответ с помощью вашей обученной LoRA-модели (если есть) или эталонного движка фактов."""
    facts = ALL_CITY_FACTS[city_name]

    if (LORA_OUTPUT_DIR / "adapter_config.json").exists() and HAS_TORCH and HAS_TRANSFORMERS and HAS_PEFT:
        import torch

        model, tokenizer = _load_trained_qwen_lora_once(LORA_OUTPUT_DIR)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_EPANET_AI},
            {
                "role": "user",
                "content": f"Вопрос инженера: {question}\n\nФакты расчёта (facts_inline):\n```json\n{json.dumps(facts, ensure_ascii=False, indent=2)}\n```",
            },
        ]
        try:
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING_MODE
            )
        except TypeError:
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        target_dev = next(model.parameters()).device
        inputs = tokenizer(prompt, return_tensors="pt").to(target_dev)
        with torch.inference_mode():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=2048,
                temperature=0.15,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.05,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
        raw_reply = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        return strip_think_tags(raw_reply), f"Обученная нейросеть {SELECTED_MODEL_ID} + QLoRA ({LORA_OUTPUT_DIR.name})"

    # Эталонный ответ по строгим фактам города (работает мгновенно ещё до завершения QLoRA)
    city_examples = build_training_examples_for_city(facts)
    q_low = question.lower()
    if "насос" in q_low or "энерг" in q_low:
        picked = next((e for e in city_examples if e["task_type"] == "pump_station_energy_audit"), city_examples[0])
    elif "скорост" in q_low or "труб" in q_low or "застой" in q_low:
        picked = next((e for e in city_examples if e["task_type"] == "pipe_velocity_and_stagnation_audit"), city_examples[0])
    elif "этаж" in q_low or "дефицит" in q_low:
        picked = next((e for e in city_examples if e["task_type"] == "pressure_and_storeys_audit"), city_examples[0])
    else:
        picked = city_examples[0]
    return picked["messages"][2]["content"], "Детерминированный эталонный интерпретатор фактов (до запуска QLoRA)"


if CITY_TO_ASK and CITY_TO_ASK in ALL_CITY_FACTS:
    print(f"Доступные города в ALL_CITY_FACTS: {list(ALL_CITY_FACTS.keys())}")
    answer_text, source_engine = generate_epanet_ai_answer(CITY_TO_ASK, USER_QUESTION)
    is_grounded, bad_numbers = verify_no_hallucinated_numbers(answer_text, ALL_CITY_FACTS[CITY_TO_ASK])
    badge_html = (
        '<span style="background:#dcfce7;color:#166534;padding:3px 10px;border-radius:999px;font-weight:700;font-size:12px;">✅ АНТИ-ГАЛЛЮЦИНАЦИОННЫЙ КОНТРОЛЬ ПРОЙДЕН (100% чисел подтверждены facts.json)</span>'
        if is_grounded
        else f'<span style="background:#fee2e2;color:#991b1b;padding:3px 10px;border-radius:999px;font-weight:700;font-size:12px;">⚠️ Найдены неподтверждённые числа: {html.escape(str(bad_numbers))}</span>'
    )
    display(HTML(
        f'<div style="font-family:Segoe UI,sans-serif;border:1px solid #cbd5e1;border-radius:10px;padding:14px 18px;background:#f8fafc;margin-bottom:10px;">'
        f'<div style="margin-bottom:6px;">{badge_html}</div>'
        f'<div style="font-size:13px;color:#334155;"><b>Город:</b> {html.escape(CITY_TO_ASK)} &bull; <b>Источник ответа:</b> {html.escape(source_engine)}</div>'
        f'</div>'
    ))
    display(Markdown(answer_text))
